# TBX11K — Object Detection Pipeline (Colab)
## 4 Architectures: FCOS, EfficientDet-D2, RetinaNet, DETR

In [ ]:
# Cell 1: Setup
import os
from google.colab import drive
drive.mount('/content/drive')

!cd /content && rm -rf /content/OBJECT_DETECTION 2>/dev/null
!mkdir -p /content/OBJECT_DETECTION
!cp /content/drive/MyDrive/OBJECT_DETECTION/images.zip /content/images.zip
!unzip -qo /content/images.zip -d /content/OBJECT_DETECTION/

# Wrap train/val/test into Images/ (zip has no Images wrapper)
if os.path.isdir('/content/OBJECT_DETECTION/train'):
    !mkdir -p /content/OBJECT_DETECTION/Images
    !mv /content/OBJECT_DETECTION/train /content/OBJECT_DETECTION/Images/
    !mv /content/OBJECT_DETECTION/val /content/OBJECT_DETECTION/Images/
    !mv /content/OBJECT_DETECTION/test /content/OBJECT_DETECTION/Images/

# If zip had Images/ wrapper instead, move contents up
if os.path.isdir('/content/OBJECT_DETECTION/Images/Images'):
    !mv /content/OBJECT_DETECTION/Images/Images/* /content/OBJECT_DETECTION/Images/
    !rmdir /content/OBJECT_DETECTION/Images/Images

# Handle lowercase 'images/' (case-sensitive filesystem)
if os.path.isdir('/content/OBJECT_DETECTION/images') and not os.path.isdir('/content/OBJECT_DETECTION/Images'):
    !mv /content/OBJECT_DETECTION/images /content/OBJECT_DETECTION/Images

# Change to project directory (build-mode safe)
%cd /content/OBJECT_DETECTION
os.makedirs('results/eda', exist_ok=True)
os.makedirs('utils', exist_ok=True)
os.makedirs('explain', exist_ok=True)

!pip install -q torch>=2.0.0 torchvision>=0.15.0 effdet timm pycocotools pillow tqdm matplotlib seaborn
!pip install -q sympy==1.13.3
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Debug: show directory structure to verify data
import subprocess
result = subprocess.run(['find', '.', '-maxdepth', '3', '-type', 'd'], capture_output=True, text=True, timeout=10)
print('\nDirectory structure (first 3 levels):')
print(result.stdout[:1500])
import os
os.environ["WANDB_MODE"] = "offline"


### 1. Write `utils/__init__.py`

In [ ]:
%%writefile utils/__init__.py
from .coco_dataset import CocoDetection, get_transform, collate_fn
from .engine import (
    train_one_epoch, evaluate, evaluate_test,
    compute_confusion_matrix, save_confusion_matrix_plot,
    set_seed, MetricTracker, save_checkpoint, load_checkpoint,
    plot_training_curves, generate_summary_report,
)
from .ema import ModelEMA


### 2. Write `utils/coco_dataset.py`

In [ ]:
%%writefile utils/coco_dataset.py
import os
from PIL import Image

import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
import torchvision.transforms as T


class CocoDetection(Dataset):
    def __init__(self, root, ann_file, transforms=None):
        self.root = root
        self.transforms = transforms
        self.coco = COCO(ann_file)
        self.ids = sorted(self.coco.imgs.keys())

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(img_id)[0]

        path = os.path.join(self.root, img_info['file_name'])
        image = Image.open(path).convert('RGB')

        boxes = []
        labels = []
        area = []
        iscrowd = []

        for ann in anns:
            bbox = ann['bbox']
            boxes.append([bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]])
            labels.append(ann['category_id'])
            area.append(ann['area'])
            iscrowd.append(ann.get('iscrowd', 0))

        if boxes:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = torch.as_tensor(area, dtype=torch.float32)
            iscrowd = torch.as_tensor(iscrowd, dtype=torch.uint8)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.uint8)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([img_id]),
            'area': area,
            'iscrowd': iscrowd,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target

    def __len__(self):
        return len(self.ids)


def get_transform(train):
    transforms_list = [T.ToTensor()]
    if train:
        transforms_list.append(T.RandomHorizontalFlip(0.5))
    transform = T.Compose(transforms_list)
    def _apply(image, target):
        return transform(image), target
    return _apply


def collate_fn(batch):
    return tuple(zip(*batch))


### 3. Write `utils/engine.py`

In [ ]:
%%writefile utils/engine.py
# =============================================================================
# Section 1: Imports, utility classes, reproducibility, metric trackers, checkpoints
# =============================================================================
import os
import json
import csv
import time
import random
import copy
import logging
from collections import defaultdict
from pathlib import Path

import torch
import torch.nn as nn
import torchvision
import numpy as np
from torchvision.ops import box_iou, nms
from scipy.optimize import linear_sum_assignment
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import wandb

CLASS_NAMES = ['Background', 'ActiveTuberculosis', 'ObsoletePulmonaryTuberculosis']
NUM_CLASSES = 3

logger = logging.getLogger(__name__)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class MetricTracker:
    def __init__(self):
        self.history = defaultdict(list)

    def update(self, **metrics):
        for k, v in metrics.items():
            self.history[k].append(v)

    def average(self, last_n=None):
        avgs = {}
        for k, vals in self.history.items():
            subset = vals[-last_n:] if last_n else vals
            avgs[k] = sum(subset) / len(subset) if subset else 0.0
        return avgs

    def best(self, metric, mode='max'):
        vals = self.history.get(metric, [])
        if not vals:
            return None, None
        idx = int(np.argmax(vals)) if mode == 'max' else int(np.argmin(vals))
        return vals[idx], idx

    def state_dict(self):
        return dict(self.history)

    def load_state_dict(self, d):
        self.history = defaultdict(list, {k: list(v) for k, v in d.items()})


def save_checkpoint(model, optimizer, epoch, metrics, path,
                    scaler=None, scheduler=None, extra=None):
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
    }
    if scaler is not None:
        state['scaler_state_dict'] = scaler.state_dict()
    if scheduler is not None:
        state['scheduler_state_dict'] = scheduler.state_dict()
    if extra is not None:
        state.update(extra)
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    torch.save(state, path)
    logger.info(f'Checkpoint saved: {path}')


def load_checkpoint(path, model, optimizer=None, scaler=None, scheduler=None):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer is not None and 'optimizer_state_dict' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scaler is not None and 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    logger.info(f'Checkpoint loaded: {path} (epoch {ckpt.get("epoch", "?")})')
    return ckpt


# =============================================================================
# Section 2: Production-grade train_one_epoch
# =============================================================================
def train_one_epoch(model, optimizer, data_loader, device, epoch,
                    scaler=None, print_freq=50, clip_norm=None,
                    metric_tracker=None, gradient_accumulation_steps=1):
    model.train()
    total_loss = 0.0
    loss_keys = None
    loss_accum = defaultdict(float)
    num_batches = 0
    start_time = time.time()
    grad_norms = []
    accum_count = 0

    optimizer.zero_grad()

    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        if scaler is not None:
            with torch.amp.autocast(device_type='cuda'):
                loss_dict = model(images, targets)
                losses = sum(loss_dict.values()) / gradient_accumulation_steps
            if not torch.isfinite(losses):
                logger.warning(f'NaN/Inf loss at step {i}, skipping batch')
                for k, v in loss_dict.items():
                    logger.warning(f'  {k}: {v.item():.6f}')
                optimizer.zero_grad()
                continue
            scaler.scale(losses).backward()
        else:
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values()) / gradient_accumulation_steps
            if not torch.isfinite(losses):
                logger.warning(f'NaN/Inf loss at step {i}, skipping batch')
                for k, v in loss_dict.items():
                    logger.warning(f'  {k}: {v.item():.6f}')
                optimizer.zero_grad()
                continue
            losses.backward()

        accum_count += 1
        total_loss += losses.item() * gradient_accumulation_steps
        num_batches += 1

        if loss_keys is None:
            loss_keys = list(loss_dict.keys())
        for k, v in loss_dict.items():
            loss_accum[k] += v.item()

        if accum_count % gradient_accumulation_steps == 0:
            if clip_norm is not None:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), clip_norm)
                grad_norms.append(grad_norm.item())
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        if i % print_freq == 0 and num_batches > 0:
            lr = optimizer.param_groups[0]['lr']
            elapsed = time.time() - start_time
            parts = [f'Epoch [{epoch}] Step [{i}/{len(data_loader)}]',
                     f'Loss: {losses.item() * gradient_accumulation_steps:.4f}',
                     f'LR: {lr:.2e}',
                     f'Time: {elapsed:.1f}s']
            if gradient_accumulation_steps > 1:
                parts.append(f'Accum: {accum_count}/{gradient_accumulation_steps}')
            print(' | '.join(parts))
            wandb.log({
                'step_loss': losses.item() * gradient_accumulation_steps,
                'step': i + epoch * len(data_loader),
                'learning_rate': lr,
            })

    avg_loss = total_loss / max(num_batches, 1)
    avg_grad_norm = float(np.mean(grad_norms)) if grad_norms else 0.0

    epoch_metrics = {'epoch_loss': avg_loss, 'avg_grad_norm': avg_grad_norm}
    if loss_keys:
        for k in loss_keys:
            epoch_metrics[f'loss_{k}'] = loss_accum[k] / max(num_batches, 1)

    wandb.log(epoch_metrics)
    if metric_tracker is not None:
        metric_tracker.update(**epoch_metrics)

    print(f'  Epoch [{epoch}] Avg Loss: {avg_loss:.4f} | '
          f'Grad Norm: {avg_grad_norm:.4f}')
    return avg_loss


# =============================================================================
# Section 3: COCO evaluation with per-class metrics, CSV, optional TTA
# =============================================================================
def _tta_forward(model, images):
    outputs = model(images)
    flipped = [torch.flip(img, dims=[-1]) for img in images]
    flipped_outputs = model(flipped)
    merged = []
    for out, f_out in zip(outputs, flipped_outputs):
        if len(f_out['boxes']) == 0:
            merged.append(out)
            continue
        img_w = float(out['boxes'][:, 2].max()) + 1.0 if len(out['boxes']) > 0 else 1.0
        f_boxes = f_out['boxes'].clone()
        f_boxes[:, [0, 2]] = img_w - f_boxes[:, [2, 0]]
        all_boxes = torch.cat([out['boxes'], f_boxes])
        all_scores = torch.cat([out['scores'], f_out['scores']])
        all_labels = torch.cat([out['labels'], f_out['labels']])
        keep = nms(all_boxes, all_scores, iou_threshold=0.5)
        merged.append({
            'boxes': all_boxes[keep], 'scores': all_scores[keep],
            'labels': all_labels[keep],
        })
    return merged


def _compute_per_class_ap(coco_gt, coco_dt, img_ids):
    per_class = {}
    cat_ids = coco_gt.getCatIds()
    for cat_id in cat_ids:
        ce = COCOeval(coco_gt, coco_dt, iouType='bbox')
        ce.params.imgIds = img_ids
        ce.params.catIds = [cat_id]
        ce.evaluate()
        ce.accumulate()
        ce.summarize()
        cat_name = coco_gt.loadCats(cat_id)[0]['name']
        per_class[f'AP_{cat_name}'] = float(ce.stats[0]) if len(ce.stats) > 0 else 0.0
    return per_class


def _append_csv(path, **kwargs):
    file_exists = os.path.exists(path)
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(kwargs.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(kwargs)


@torch.no_grad()
def evaluate(model, data_loader, device, output_file=None, tta=False):
    model.eval()
    coco_gt = data_loader.dataset.coco
    results = []
    img_ids = []
    _debug_done = False

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = _tta_forward(model, images) if tta else model(images)

        if not _debug_done:
            for bi in range(len(outputs)):
                out = outputs[bi]
                tgt = targets[bi]
                n_pred = len(out['boxes'])
                n_gt = len(tgt['boxes'])
                if n_pred > 0 and n_gt > 0:
                    print(f'[DEBUG] batch_img={bi} n_pred={n_pred} n_gt={n_gt}')
                    print(f'[DEBUG] pred_boxes_range: x[{out["boxes"][:,0].min():.1f}-{out["boxes"][:,0].max():.1f}] '
                          f'y[{out["boxes"][:,1].min():.1f}-{out["boxes"][:,1].max():.1f}]')
                    print(f'[DEBUG] gt_boxes_range:  x[{tgt["boxes"][:,0].min():.1f}-{tgt["boxes"][:,0].max():.1f}] '
                          f'y[{tgt["boxes"][:,1].min():.1f}-{tgt["boxes"][:,1].max():.1f}]')
                    print(f'[DEBUG] pred_labels: {out["labels"][:10].cpu().numpy()}')
                    print(f'[DEBUG] gt_labels:  {tgt["labels"][:10].numpy()}')
                    print(f'[DEBUG] pred_scores: {out["scores"][:5].cpu().numpy()}')
                    print(f'[DEBUG] pred_boxes[:3]: {out["boxes"][:3].cpu().numpy()}')
                    print(f'[DEBUG] gt_boxes[:3]: {tgt["boxes"][:3].numpy()}')
                    ious = box_iou(tgt['boxes'], out['boxes'][:20]).numpy()
                    print(f'[DEBUG] max_iou_per_pred: {ious.max(axis=0)[:5]}')
                    print(f'[DEBUG] max_iou_per_gt:  {ious.max(axis=1)[:5]}')
                    _debug_done = True
                    break

        for target, output in zip(targets, outputs):
            img_id = target['image_id'].item()
            img_ids.append(img_id)
            boxes = output['boxes'].cpu().numpy()
            scores = output['scores'].cpu().numpy()
            labels = output['labels'].cpu().numpy()

            keep = scores >= 0.05
            boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

            if len(boxes) > 0:
                keep_idx = nms(torch.tensor(boxes), torch.tensor(scores),
                               iou_threshold=0.5)
                boxes, scores, labels = (boxes[keep_idx], scores[keep_idx],
                                         labels[keep_idx])
                boxes = np.atleast_2d(boxes)
                scores = np.atleast_1d(scores)
                labels = np.atleast_1d(labels)

            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box
                w, h = x2 - x1, y2 - y1
                results.append({
                    'image_id': img_id,
                    'bbox': [float(x1), float(y1), float(w), float(h)],
                    'score': float(score),
                    'category_id': int(label),
                })

    if not results:
        print('[EVAL] No predictions, skipping COCOeval.')
        return None

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.params.imgIds = sorted(set(img_ids))
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    stats = coco_eval.stats.tolist()
    metrics = {
        'mAP@0.5:0.95': stats[0], 'mAP@0.5': stats[1], 'mAP@0.75': stats[2],
        'mAP_small': stats[3], 'mAP_medium': stats[4], 'mAP_large': stats[5],
        'AR@1': stats[6], 'AR@10': stats[7], 'AR@100': stats[8],
    }

    per_class = _compute_per_class_ap(coco_gt, coco_dt, sorted(set(img_ids)))
    metrics.update(per_class)

    if output_file:
        os.makedirs(os.path.dirname(output_file) or '.', exist_ok=True)
        with open(output_file, 'w') as f:
            json.dump({'metrics': metrics, 'per_class': per_class,
                       'predictions': results}, f, indent=2)
        csv_path = output_file.replace('.json', '_log.csv')
        _append_csv(csv_path, epoch=0, **metrics)
        print(f'[EVAL] Saved {output_file} and {csv_path}')

    return coco_eval


@torch.no_grad()
def evaluate_test(model, data_loader, device, output_file, model_name="unknown"):
    import datetime
    model.eval()
    results = []

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for target, output in zip(targets, outputs):
            img_id = target['image_id'].item()
            boxes = output['boxes'].cpu().numpy()
            scores = output['scores'].cpu().numpy()
            labels = output['labels'].cpu().numpy()

            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box
                w, h = x2 - x1, y2 - y1
                results.append({
                    'image_id': img_id,
                    'bbox': [float(x1), float(y1), float(w), float(h)],
                    'score': float(score),
                    'category_id': int(label),
                })

    output = {
        'model': model_name,
        'dataset': 'TBX11K',
        'num_predictions': len(results),
        'date': datetime.datetime.now().isoformat(),
        'predictions': results,
    }

    os.makedirs(os.path.dirname(output_file) or '.', exist_ok=True)
    with open(output_file, 'w') as f:
        json.dump(output, f, indent=2)
    print(f'[TEST] {len(results)} predictions saved to {output_file}')


# =============================================================================
# Section 4: Hungarian-matching confusion matrix with per-class metrics
# =============================================================================
@torch.no_grad()
def compute_confusion_matrix(model, data_loader, device,
                             iou_thresh=0.5, score_thresh=0.05):
    model.eval()
    confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        outputs = model(images)

        for pred, target in zip(outputs, targets):
            gt_boxes = target['boxes'].cpu()
            gt_labels = target['labels'].cpu()

            keep = pred['scores'] >= score_thresh
            pred_boxes = pred['boxes'][keep].cpu()
            pred_labels = pred['labels'][keep].cpu()

            matched_gt = set()
            matched_pred = set()

            if len(gt_boxes) > 0 and len(pred_boxes) > 0:
                iou_matrix = box_iou(gt_boxes, pred_boxes).numpy()
                cost_matrix = np.where(iou_matrix >= iou_thresh,
                                       iou_matrix, 0.0)

                if cost_matrix.sum() > 0:
                    gt_idx, pred_idx = linear_sum_assignment(
                        cost_matrix, maximize=True)
                    for gi, pi in zip(gt_idx, pred_idx):
                        if iou_matrix[gi, pi] >= iou_thresh:
                            gt_cls = gt_labels[gi].item()
                            pred_cls = pred_labels[pi].item()
                            confusion[gt_cls, pred_cls] += 1
                            matched_gt.add(gi)
                            matched_pred.add(pi)

            for gi in range(len(gt_boxes)):
                if gi not in matched_gt:
                    confusion[gt_labels[gi].item(), 0] += 1

            for pi in range(len(pred_boxes)):
                if pi not in matched_pred:
                    confusion[0, pred_labels[pi].item()] += 1

    return confusion


def save_confusion_matrix_plot(confusion, output_path, class_names=None):
    if class_names is None:
        class_names = CLASS_NAMES

    fig, ax = plt.subplots(figsize=(8, 7))
    total_per_row = confusion.sum(axis=1, keepdims=True)
    total_per_row = np.where(total_per_row == 0, 1, total_per_row)
    confusion_norm = confusion / total_per_row

    sns.heatmap(confusion_norm, annot=confusion, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, cbar_kws={'label': 'Fraction'})
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title('Confusion Matrix (Hungarian Match, IoU >= 0.5)')

    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f'[CONFUSION] Saved to {output_path}')

    print('\n--- Per-Class Metrics (Hungarian) ---')
    results = {}
    for cls_idx in range(1, NUM_CLASSES):
        tp = int(confusion[cls_idx, cls_idx])
        fp = int(confusion[:, cls_idx].sum() - tp)
        fn = int(confusion[cls_idx, :].sum() - tp)
        tn = int(confusion.sum() - tp - fp - fn)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0.0)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        balanced_acc = (recall + specificity) / 2.0

        name = class_names[cls_idx] if cls_idx < len(class_names) else f'Class{cls_idx}'
        print(f'  {name}: P={precision:.3f} R={recall:.3f} F1={f1:.3f} '
              f'Spec={specificity:.3f} BalAcc={balanced_acc:.3f}')
        results[name] = {
            'precision': precision, 'recall': recall, 'f1': f1,
            'specificity': specificity, 'balanced_accuracy': balanced_acc,
        }

    return confusion, results


# =============================================================================
# Section 5: Visualization utilities and reporting helpers
# =============================================================================
def plot_training_curves(metrics, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if 'epoch_loss' in metrics.history:
        axes[0].plot(metrics.history['epoch_loss'], marker='o', color='#4C72B0')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Loss')
        axes[0].grid(True, alpha=0.3)

    if 'learning_rate' in metrics.history:
        axes[1].plot(metrics.history['learning_rate'], marker='s', color='#DD8452')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Learning Rate')
        axes[1].set_title('Learning Rate Schedule')
        axes[1].set_yscale('log')
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/training_curves.png', dpi=150)
    plt.close()
    print(f'[VIZ] Training curves saved to {output_dir}/training_curves.png')


def generate_summary_report(all_metrics, output_path):
    lines = ['=' * 70,
             'TBX11K Object Detection - Model Comparison Report',
             '=' * 70, '']
    for model_name, m in all_metrics.items():
        lines.append(f'--- {model_name.upper()} ---')
        for k, v in m.items():
            if isinstance(v, float):
                lines.append(f'  {k}: {v:.4f}')
            else:
                lines.append(f'  {k}: {v}')
        lines.append('')

    report = '\n'.join(lines)
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w') as f:
        f.write(report)
    print(f'[REPORT] Saved to {output_path}')
    return report


### 4. Write `utils/ema.py`

In [ ]:
%%writefile utils/ema.py
import copy
import torch
import torch.nn as nn


class ModelEMA:
    """Exponential Moving Average of model weights.

    During training, call `ema.update(model)` after each optimizer step.
    At evaluation, use `ema.model` instead of the raw model.
    """

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.model = copy.deepcopy(model)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        decay = self.decay
        with torch.no_grad():
            for ema_p, model_p in zip(self.model.parameters(), model.parameters()):
                ema_p.mul_(decay).add_(model_p, alpha=1.0 - decay)
            for ema_b, model_b in zip(self.model.buffers(), model.buffers()):
                ema_b.copy_(model_b)

    def state_dict(self):
        return self.model.state_dict()

    def load_state_dict(self, state_dict):
        self.model.load_state_dict(state_dict)


### 5. Write `explain/__init__.py`

In [ ]:
%%writefile explain/__init__.py
from .gradcam import GradCAM, get_target_layer
from .detr_attention import DETRAttentionExtractor
from .visualize import overlay_heatmap, draw_detections, generate_xai_report


### 6. Write `explain/gradcam.py`

In [ ]:
%%writefile explain/gradcam.py
import torch
import torch.nn.functional as F
import numpy as np


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self._activations = None
        self._gradients = None

        self._forward_handle = target_layer.register_forward_hook(self._hook_forward)
        self._backward_handle = target_layer.register_full_backward_hook(self._hook_backward)

    def _hook_forward(self, module, input, output):
        self._activations = output.detach()

    def _hook_backward(self, module, grad_input, grad_output):
        self._gradients = grad_output[0].detach()

    def remove_hooks(self):
        self._forward_handle.remove()
        self._backward_handle.remove()

    def generate(self, image, target_class=None):
        self.model.zero_grad()

        if image.grad is not None:
            image.grad.zero_()

        predictions = self.model(image)

        if target_class is None:
            if len(predictions[0]['scores']) == 0:
                return None
            best_idx = predictions[0]['scores'].argmax()
            target_class = predictions[0]['labels'][best_idx].item()

        loss = torch.tensor(0.0, device=image.device)
        for pred in predictions:
            for label, score in zip(pred['labels'], pred['scores']):
                if label.item() == target_class:
                    loss = loss + score

        if loss == 0:
            return None

        loss.backward()

        if self._gradients is None or self._activations is None:
            return None

        weights = self._gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self._activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def get_target_layer(model, architecture):
    if architecture == 'fcos':
        return model.backbone.body.layer4
    elif architecture == 'retinanet':
        return model.backbone.body.layer4
    elif architecture == 'efficientdet':
        return model.backbone.conv_head
    else:
        raise ValueError(f'Unsupported architecture: {architecture}')


### 7. Write `explain/detr_attention.py`

In [ ]:
%%writefile explain/detr_attention.py
import torch
import torch.nn.functional as F
import numpy as np


class DETRAttentionExtractor:
    def __init__(self, model):
        self.model = model
        self._attention_weights = None

        decoder_layer = model.transformer.decoder.layers[-1]
        self._handle = decoder_layer.multihead_attn.register_forward_hook(
            self._hook_attention
        )

    def _hook_attention(self, module, input, output):
        q = input[0]
        k = input[1]
        attn_weights = torch.bmm(q.transpose(0, 1), k.transpose(0, 1).transpose(1, 2))
        d_k = q.size(-1) ** 0.5
        attn_weights = F.softmax(attn_weights / d_k, dim=-1)
        self._attention_weights = attn_weights.detach()

    def remove_hook(self):
        self._handle.remove()

    def extract(self, image):
        self.model.zero_grad()

        with torch.no_grad():
            output = self.model(image)

        if self._attention_weights is None:
            return None, output

        attn = self._attention_weights

        attn = attn.mean(dim=0)

        num_queries = attn.shape[0]

        pred_boxes = output[0]['boxes'].cpu().numpy()
        pred_scores = output[0]['scores'].cpu().numpy()
        pred_labels = output[0]['labels'].cpu().numpy()

        heatmaps = []
        meta = []
        for q_idx in range(min(num_queries, 100)):
            if q_idx >= len(pred_scores):
                continue
            if pred_scores[q_idx] < 0.1:
                continue

            attn_map = attn[q_idx].reshape(-1).cpu().numpy()
            h = w = int(np.sqrt(len(attn_map)))

            if h * w != len(attn_map):
                h = w = 32

            attn_map = attn_map[:h * w].reshape(h, w)
            attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

            heatmaps.append(attn_map)
            meta.append({
                'query_idx': q_idx,
                'score': float(pred_scores[q_idx]),
                'label': int(pred_labels[q_idx]),
                'box': pred_boxes[q_idx].tolist(),
            })

        return heatmaps, meta


def get_attention_map(detr_model, image, output_size=(512, 512)):
    extractor = DETRAttentionExtractor(detr_model)
    heatmaps, meta = extractor.extract(image)
    extractor.remove_hook()

    if heatmaps is None or len(heatmaps) == 0:
        return None, None

    return heatmaps, meta


### 8. Write `explain/visualize.py`

In [ ]:
%%writefile explain/visualize.py
import os
import numpy as np
from PIL import Image, ImageDraw

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CLASS_NAMES = {0: 'Background', 1: 'ActiveTuberculosis', 2: 'ObsoletePulmonaryTuberculosis'}
CLASS_COLORS = {1: (255, 0, 0), 2: (0, 200, 255)}


def overlay_heatmap(image_tensor, heatmap, alpha=0.6):
    if heatmap is None:
        img_np = image_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        return Image.fromarray(img_np)

    img_np = image_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

    cmap = plt.get_cmap('jet')
    heatmap_colored = cmap(heatmap)[:, :, :3]
    heatmap_colored = (heatmap_colored * 255).astype(np.uint8)

    blended = (1 - alpha) * img_np + alpha * heatmap_colored
    blended = np.clip(blended, 0, 255).astype(np.uint8)
    return Image.fromarray(blended)


def draw_detections(pil_image, predictions, class_names=None, score_thresh=0.3):
    if class_names is None:
        class_names = CLASS_NAMES
    draw = ImageDraw.Draw(pil_image)

    boxes = predictions.get('boxes', [])
    scores = predictions.get('scores', [])
    labels = predictions.get('labels', [])

    for box, score, label in zip(boxes, scores, labels):
        if score < score_thresh:
            continue
        x1, y1, x2, y2 = box[:4]
        color = CLASS_COLORS.get(int(label), (255, 255, 0))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        label_text = f'{class_names.get(int(label), str(label))}: {score:.2f}'
        draw.text((x1 + 2, y1 - 16), label_text, fill=color)

    return pil_image


def generate_xai_report(model_outputs, image_tensor, ground_truth, output_path,
                        arch_name, attention_data=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    img_np = image_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
    base_img = Image.fromarray(img_np)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img_np)
    axes[0].set_title(f'{arch_name} — Original + Detections', fontsize=11)
    if model_outputs:
        draw = ImageDraw.Draw(base_img)
        boxes = model_outputs[0]['boxes'].cpu().numpy()
        scores = model_outputs[0]['scores'].cpu().numpy()
        labels = model_outputs[0]['labels'].cpu().numpy()
        for box, sc, lb in zip(boxes, scores, labels):
            if sc < 0.3:
                continue
            x1, y1, x2, y2 = box
            color = CLASS_COLORS.get(int(lb), (255, 255, 0))
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        axes[0].imshow(np.array(base_img))

    if attention_data is not None:
        heatmaps, meta = attention_data
        if heatmaps and len(heatmaps) > 0:
            best_idx = 0
            best_score = 0
            for i, m in enumerate(meta):
                if m['score'] > best_score:
                    best_score = m['score']
                    best_idx = i

            if best_idx < len(heatmaps):
                attn_resized = np.array(Image.fromarray(heatmaps[best_idx]).resize(
                    (img_np.shape[1], img_np.shape[0]), Image.BILINEAR))
                cmap = plt.get_cmap('jet')
                attn_colored = cmap(attn_resized)[:, :, :3]
                blended = (1 - 0.5) * img_np / 255 + 0.5 * attn_colored
                axes[1].imshow(np.clip(blended, 0, 1))
                axes[1].set_title(f'{arch_name} — Attention (score={best_score:.2f})', fontsize=11)
            else:
                axes[1].imshow(img_np)
                axes[1].set_title(f'{arch_name} — No attention', fontsize=11)
        else:
            axes[1].imshow(img_np)
            axes[1].set_title(f'{arch_name} — No attention', fontsize=11)
    else:
        axes[1].imshow(img_np)
        axes[1].set_title(f'{arch_name} — No attention', fontsize=11)

    axes[2].imshow(img_np)
    ax2_title = 'Ground Truth'
    if ground_truth is not None and len(ground_truth.get('boxes', [])) > 0:
        draw_gt = ImageDraw.Draw(Image.fromarray(img_np.copy()))
        for box, label in zip(ground_truth['boxes'], ground_truth['labels']):
            x1, y1, x2, y2 = box
            color = CLASS_COLORS.get(int(label), (255, 255, 0))
            draw_gt.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw_gt.text((x1 + 2, y1 - 16), CLASS_NAMES.get(int(label), str(label)), fill=color)
    axes[2].imshow(img_np)
    axes[2].set_title(ax2_title, fontsize=11)

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()


def save_comparison_grid(model_results, output_path, class_names=None):
    if class_names is None:
        class_names = ['FCOS', 'EfficientDet-D2', 'RetinaNet', 'DETR']

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for idx, (name, img_path) in enumerate(model_results.items()):
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[idx].imshow(np.array(img))
        axes[idx].set_title(name, fontsize=12)
        axes[idx].axis('off')

    plt.suptitle('Model Comparison — XAI Overlays', fontsize=14)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[XAI] Comparison grid saved to {output_path}')


### 9. Write `convert.py`

In [ ]:
%%writefile convert.py
import os
import json
import glob
import shutil
import argparse

from PIL import Image

DATA_ROOT = 'Images'
OUTPUT_DIR = 'dataset'

CLASS_MAP = {
    'ActiveTuberculosis': 1,
    'ObsoletePulmonaryTuberculosis': 2,
}
CLASS_NAMES = ['Background', 'ActiveTuberculosis', 'ObsoletePulmonaryTuberculosis']


def make_dir(path):
    os.makedirs(path, exist_ok=True)


def convert_to_coco(splits):
    for split in splits:
        images = []
        annotations = []
        ann_id = 1

        ann_dir = os.path.join(DATA_ROOT, split, 'ann')
        img_dir = os.path.join(DATA_ROOT, split, 'img')
        out_img_dir = os.path.join(OUTPUT_DIR, 'coco', split)
        make_dir(out_img_dir)

        for f in sorted(glob.glob(os.path.join(ann_dir, '*.json'))):
            with open(f) as fh:
                d = json.load(fh)

            basename = os.path.basename(f).replace('.json', '')
            img_id = len(images) + 1

            src_path = os.path.join(img_dir, basename)
            if not os.path.exists(src_path):
                print(f'[WARN] Image not found: {src_path}, skipping')
                continue

            dst_path = os.path.join(out_img_dir, basename)
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)

            w_img = d['size']['width']
            h_img = d['size']['height']

            images.append({
                'id': img_id,
                'file_name': basename,
                'width': w_img,
                'height': h_img,
            })

            for obj in d.get('objects', []):
                if obj.get('classTitle') not in CLASS_MAP:
                    continue
                ext = obj['points']['exterior']
                if len(ext) < 2:
                    continue
                x1, y1 = ext[0]
                x2, y2 = ext[1]
                x_min, x_max = min(x1, x2), max(x1, x2)
                y_min, y_max = min(y1, y2), max(y1, y2)
                w = x_max - x_min
                h = y_max - y_min
                if w <= 0 or h <= 0:
                    continue
                x_min = max(0, min(x_min, w_img))
                y_min = max(0, min(y_min, h_img))
                w = min(w, w_img - x_min)
                h = min(h, h_img - y_min)
                if w <= 0 or h <= 0:
                    continue

                annotations.append({
                    'id': ann_id,
                    'image_id': img_id,
                    'bbox': [float(x_min), float(y_min), float(w), float(h)],
                    'area': float(w * h),
                    'category_id': CLASS_MAP[obj['classTitle']],
                    'iscrowd': 0,
                })
                ann_id += 1

        out = {
            'images': images,
            'annotations': annotations,
            'categories': [
                {'id': 1, 'name': 'ActiveTuberculosis'},
                {'id': 2, 'name': 'ObsoletePulmonaryTuberculosis'},
            ],
        }

        out_path = os.path.join(OUTPUT_DIR, 'coco', f'{split}.json')
        with open(out_path, 'w') as f:
            json.dump(out, f)
        print(f'[COCO] {split}: {len(images)} images, {len(annotations)} annotations → {out_path}')


def main():
    parser = argparse.ArgumentParser(description='Convert TBX11K annotations to COCO format')
    parser.add_argument('--data-root', type=str, default='Images', help='Path to Images directory')
    parser.add_argument('--output-dir', type=str, default='dataset', help='Output directory')
    args = parser.parse_args()

    global DATA_ROOT, OUTPUT_DIR
    DATA_ROOT = args.data_root
    OUTPUT_DIR = args.output_dir

    make_dir(OUTPUT_DIR)

    convert_to_coco(['train', 'val'])

    print('\nConversion complete.')
    print(f'  COCO: {os.path.join(OUTPUT_DIR, "coco")}/')


if __name__ == '__main__':
    main()


### 10. Write `eda.py`

In [ ]:
%%writefile eda.py
import os
import json
import glob
from collections import Counter, defaultdict
from statistics import mean, median

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

DATA_ROOT = 'Images'
OUTPUT_DIR = 'results/eda'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_MAP = {
    'ActiveTuberculosis': 1,
    'ObsoletePulmonaryTuberculosis': 2,
}
CLASS_NAMES = ['Background', 'ActiveTuberculosis', 'ObsoletePulmonaryTuberculosis']


def load_annotations(splits=None):
    if splits is None:
        splits = ['train', 'val', 'test']
    records = []
    for split in splits:
        ann_dir = os.path.join(DATA_ROOT, split, 'ann')
        img_dir = os.path.join(DATA_ROOT, split, 'img')
        for f in sorted(glob.glob(os.path.join(ann_dir, '*.json'))):
            with open(f) as fh:
                d = json.load(fh)
            basename = os.path.basename(f).replace('.png.json', '')
            tags = [t['name'] for t in d.get('tags', [])]
            objs = []
            for o in d.get('objects', []):
                ext = o['points']['exterior']
                x1, y1 = ext[0]
                x2, y2 = ext[1]
                w, h = x2 - x1, y2 - y1
                objs.append({
                    'class': o['classTitle'],
                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'w': w, 'h': h,
                    'cx': (x1 + x2) / 2,
                    'cy': (y1 + y2) / 2,
                    'area': w * h,
                })
            records.append({
                'split': split,
                'img_id': basename,
                'img_path': os.path.join(img_dir, f'{basename}.png'),
                'tags': tags,
                'objs': objs,
                'width': d['size']['width'],
                'height': d['size']['height'],
            })
    return records


def plot_tag_distribution(records):
    splits = ['train', 'val', 'test']
    tag_order = ['healthy', 'sick_but_non-tb', 'active_tb', 'latent_tb', 'active&latent_tb']
    data = {}
    for sp in splits:
        counter = Counter()
        for r in records:
            if r['split'] == sp:
                for t in r['tags']:
                    counter[t] += 1
        data[sp] = [counter.get(t, 0) for t in tag_order]

    x = np.arange(len(tag_order))
    w = 0.25
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#4C72B0', '#DD8452', '#55A868']
    for i, sp in enumerate(splits):
        bars = ax.bar(x + i * w, data[sp], w, label=sp.upper(), color=colors[i])
        for bar, val in zip(bars, data[sp]):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                        str(val), ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x + w)
    ax.set_xticklabels([t.replace('_', '\n') for t in tag_order])
    ax.set_ylabel('Image count')
    ax.set_title('Image-level Tag Distribution per Split')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'tag_distribution.png'))
    plt.close()
    print('[EDA] Saved tag_distribution.png')


def plot_bbox_class_distribution(records):
    splits = ['train', 'val']
    classes = ['ActiveTuberculosis', 'ObsoletePulmonaryTuberculosis']
    data = {}
    for sp in splits:
        counter = Counter()
        for r in records:
            if r['split'] == sp:
                for o in r['objs']:
                    counter[o['class']] += 1
        data[sp] = [counter.get(c, 0) for c in classes]

    x = np.arange(len(classes))
    w = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#4C72B0', '#DD8452']
    for i, sp in enumerate(splits):
        bars = ax.bar(x + i * w, data[sp], w, label=sp.upper(), color=colors[i])
        for bar, val in zip(bars, data[sp]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                    str(val), ha='center', va='bottom', fontsize=10)

    ax.set_xticks(x + w / 2)
    ax.set_xticklabels(classes, rotation=15, ha='right')
    ax.set_ylabel('Bounding box count')
    ax.set_title('Object Class Distribution')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'bbox_class_distribution.png'))
    plt.close()
    print('[EDA] Saved bbox_class_distribution.png')


def plot_bbox_spatial_heatmap(records):
    all_cx = []
    all_cy = []
    for r in records:
        if r['split'] == 'test':
            continue
        for o in r['objs']:
            all_cx.append(o['cx'] / r['width'])
            all_cy.append(o['cy'] / r['height'])

    if not all_cx:
        return

    fig, ax = plt.subplots(figsize=(6, 6))
    heatmap, xedges, yedges = np.histogram2d(all_cx, all_cy, bins=32,
                                              range=[[0, 1], [0, 1]])
    ax.imshow(heatmap.T, origin='lower', cmap='hot', interpolation='bilinear',
              extent=[0, 1, 0, 1])
    ax.set_xlabel('Normalized X (left → right)')
    ax.set_ylabel('Normalized Y (top → bottom)')
    ax.set_title('Bounding Box Center Spatial Distribution\n(Train + Val)')
    plt.colorbar(plt.cm.ScalarMappable(cmap='hot'), ax=ax, label='Count')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'bbox_spatial_heatmap.png'))
    plt.close()
    print('[EDA] Saved bbox_spatial_heatmap.png')

    print(f'  Box centers: {len(all_cx)} total')
    print(f'  Mean center: ({mean(all_cx):.3f}, {mean(all_cy):.3f})')
    print(f'  Top half: {sum(1 for cy in all_cy if cy < 0.5)} / {len(all_cy)} ({sum(1 for cy in all_cy if cy < 0.5)/len(all_cy)*100:.1f}%)')


def plot_bbox_size_distribution(records):
    widths = []
    heights = []
    for r in records:
        if r['split'] == 'test':
            continue
        for o in r['objs']:
            widths.append(o['w'])
            heights.append(o['h'])

    if not widths:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.scatter(widths, heights, alpha=0.4, s=15, c='#4C72B0', edgecolors='none')
    ax.set_xlabel('Width (px)')
    ax.set_ylabel('Height (px)')
    ax.set_title(f'Box Size Scatter (n={len(widths)})')
    ax.axhline(mean(heights), color='red', linestyle='--', alpha=0.6, label=f'Mean H={mean(heights):.0f}')
    ax.axvline(mean(widths), color='green', linestyle='--', alpha=0.6, label=f'Mean W={mean(widths):.0f}')
    ax.legend(fontsize=9)

    ax = axes[1]
    rel_areas = [(w * h) / (512 * 512) * 100 for w, h in zip(widths, heights)]
    ax.hist(rel_areas, bins=40, color='#DD8452', edgecolor='white', alpha=0.8)
    ax.axvline(mean(rel_areas), color='red', linestyle='--', alpha=0.7,
               label=f'Mean={mean(rel_areas):.2f}%')
    ax.set_xlabel('Relative area (% of image)')
    ax.set_ylabel('Count')
    ax.set_title('Box Area Distribution')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'bbox_size_distribution.png'))
    plt.close()
    print('[EDA] Saved bbox_size_distribution.png')


def plot_bbox_aspect_ratio(records):
    ratios = []
    for r in records:
        if r['split'] == 'test':
            continue
        for o in r['objs']:
            if o['h'] > 0:
                ratios.append(o['w'] / o['h'])

    if not ratios:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(ratios, bins=30, color='#55A868', edgecolor='white', alpha=0.8)
    ax.axvline(mean(ratios), color='red', linestyle='--', alpha=0.7,
               label=f'Mean={mean(ratios):.2f}')
    ax.axvline(median(ratios), color='blue', linestyle=':', alpha=0.7,
               label=f'Median={median(ratios):.2f}')
    ax.axvline(1.0, color='gray', linestyle='-', alpha=0.4, label='Square (1:1)')
    ax.set_xlabel('Aspect Ratio (width / height)')
    ax.set_ylabel('Count')
    ax.set_title('Bounding Box Aspect Ratio Distribution')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'bbox_aspect_ratio.png'))
    plt.close()
    print('[EDA] Saved bbox_aspect_ratio.png')

    print(f'  Aspect ratio: min={min(ratios):.2f}, max={max(ratios):.2f}, mean={mean(ratios):.2f}, median={median(ratios):.2f}')


def plot_boxes_per_image(records):
    splits = ['train', 'val']
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for idx, sp in enumerate(splits):
        counter = Counter()
        for r in records:
            if r['split'] == sp:
                counter[len(r['objs'])] += 1
        max_bins = max(counter.keys()) + 1
        bins = list(range(max_bins + 1))
        counts = [counter.get(i, 0) for i in range(max_bins + 1)]

        ax = axes[idx]
        ax.bar(bins, counts, color='#4C72B0', edgecolor='white', width=0.8)
        ax.set_xlabel('Number of bounding boxes')
        ax.set_ylabel('Image count')
        ax.set_title(f'{sp.upper()} — Boxes per Image')
        ax.set_xticks(bins)
        for i, c in enumerate(counts):
            if c > 0:
                ax.text(i, c + 2, str(c), ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'boxes_per_image.png'))
    plt.close()
    print('[EDA] Saved boxes_per_image.png')


def plot_sample_grid(records):
    pos_samples = [r for r in records if r['objs'] and r['split'] != 'test']
    if len(pos_samples) > 16:
        import random
        random.seed(42)
        pos_samples = random.sample(pos_samples, 16)

    n = len(pos_samples)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.flatten() if n > 1 else [axes]

    for i, r in enumerate(pos_samples):
        img = Image.open(r['img_path']).convert('RGB')
        draw = ImageDraw.Draw(img, 'RGBA')
        for o in r['objs']:
            color = (255, 0, 0, 120) if o['class'] == 'ActiveTuberculosis' else (0, 255, 170, 120)
            draw.rectangle([o['x1'], o['y1'], o['x2'], o['y2']], outline=color[:3], width=3)
            label = 'ATB' if o['class'] == 'ActiveTuberculosis' else 'OTB'
            draw.text((o['x1'] + 2, o['y1'] - 14), label, fill=color[:3])
        axes[i].imshow(np.array(img))
        axes[i].set_title(f'{r["split"]}: {", ".join(r["tags"])}', fontsize=9)
        axes[i].axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Sample Images with Bounding Boxes\n(Red=ActiveTB, Cyan=ObsoleteTB)', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'sample_grid.png'))
    plt.close()
    print('[EDA] Saved sample_grid.png')


def print_dataset_summary(records):
    total = len(records)
    splits = Counter(r['split'] for r in records)
    total_with_boxes = sum(1 for r in records if r['objs'])
    total_boxes = sum(len(r['objs']) for r in records)

    tag_counter = Counter()
    box_class_counter = Counter()
    for r in records:
        for t in r['tags']:
            tag_counter[t] += 1
        for o in r['objs']:
            box_class_counter[o['class']] += 1

    lines = [
        '=' * 60,
        'DATASET SUMMARY — TBX11K',
        '=' * 60,
        f'Total images:         {total}',
        f'  Train:              {splits.get("train", 0)}',
        f'  Val:                {splits.get("val", 0)}',
        f'  Test:               {splits.get("test", 0)}',
        f'Image size:           512 × 512',
        '',
        f'Images with boxes:    {total_with_boxes} ({total_with_boxes/total*100:.1f}%)',
        f'Total bounding boxes: {total_boxes}',
        '',
        'Image-level tag distribution:',
    ]
    for tag, count in tag_counter.most_common():
        lines.append(f'  {tag}: {count} ({count/total*100:.1f}%)')

    lines.extend([
        '',
        'Bounding box class distribution:',
    ])
    for cls_name, count in box_class_counter.most_common():
        lines.append(f'  {cls_name}: {count} ({count/total_boxes*100:.1f}%)')

    lines.extend([
        '',
        'Tag → Box mapping (train+val):',
        '  active_tb         → 100% ActiveTuberculosis',
        '  latent_tb         → 100% ObsoletePulmonaryTuberculosis',
        '  active&latent_tb  → Both classes (~50/50)',
        '  healthy           → No boxes',
        '  sick_but_non-tb   → No boxes',
        '',
        'Test set: 3302 images with no tags and no boxes — inference only.',
        '=' * 60,
    ])

    summary = '\n'.join(lines)
    print(summary)
    with open(os.path.join(OUTPUT_DIR, 'dataset_summary.txt'), 'w') as f:
        f.write(summary)
    print('[EDA] Saved dataset_summary.txt')


def main():
    print('Loading annotations...')
    records = load_annotations()
    print(f'Loaded {len(records)} annotation files.\n')

    print_dataset_summary(records)
    print()
    plot_tag_distribution(records)
    plot_bbox_class_distribution(records)
    plot_bbox_spatial_heatmap(records)
    plot_bbox_size_distribution(records)
    plot_bbox_aspect_ratio(records)
    plot_boxes_per_image(records)
    plot_sample_grid(records)

    print(f'\nAll EDA outputs saved to {OUTPUT_DIR}/')


if __name__ == '__main__':
    main()


### 11. Write `train_fcos.py`

In [ ]:
%%writefile train_fcos.py
#!/usr/bin/env python3
"""FCOS Training Pipeline — Research-Grade Implementation for TBX11K.

Usage:
    python train_fcos.py
    python train_fcos.py --epochs 50 --lr 0.001
    python train_fcos.py --resume results/fcos/weights/last_checkpoint.pth
    python train_fcos.py --optimizer AdamW --batch-size 4
"""

import os
import sys
import json
import copy
import argparse

import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models.detection import fcos_resnet50_fpn
import torchvision.transforms as T
import torchvision.transforms.functional as TF

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass
from utils.coco_dataset import CocoDetection, collate_fn
from utils.engine import (
    set_seed, MetricTracker, save_checkpoint, load_checkpoint,
    train_one_epoch, evaluate, evaluate_test,
    compute_confusion_matrix, save_confusion_matrix_plot,
    plot_training_curves,
)
from utils.ema import ModelEMA
from explain.gradcam import GradCAM, get_target_layer
from explain.visualize import overlay_heatmap, draw_detections

# =============================================================================
# Configuration
# =============================================================================
DEFAULT_CONFIG = {
    "model": {
        "name": "FCOS-ResNet50-FPN",
        "num_classes": 3,
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 0.005,
        "optimizer": "SGD",
        "weight_decay": 1e-4,
        "momentum": 0.9,
        "clip_norm": 1.0,
        "warmup_epochs": 5,
        "ema_decay": 0.99,
        "early_stop_patience": 15,
        "save_every": 10,
        "seed": 42,
        "resume": None,
    },
    "augmentation": {
        "hflip_prob": 0.5,
        "brightness": 0.3,
        "contrast": 0.3,
        "gamma": 0.2,
        "noise_std": 0.05,
        "normalize": True,
    },
    "data": {
        "train_images": "dataset/coco/train",
        "val_images": "dataset/coco/val",
        "train_ann": "dataset/coco/train.json",
        "val_ann": "dataset/coco/val.json",
        "test_images": "dataset/coco/test",
        "test_ann": "dataset/coco/test.json",
        "num_workers": 2,
    },
    "results_dir": "results/fcos",
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# =============================================================================
# CLI Argument Parser
# =============================================================================
def parse_args():
    p = argparse.ArgumentParser(description="FCOS TB Detection Training")
    p.add_argument("--epochs", type=int, default=None)
    p.add_argument("--batch-size", type=int, default=None)
    p.add_argument("--lr", type=float, default=None)
    p.add_argument("--optimizer", type=str, default=None, choices=["SGD", "AdamW"])
    p.add_argument("--weight-decay", type=float, default=None)
    p.add_argument("--momentum", type=float, default=None)
    p.add_argument("--clip-norm", type=float, default=None)
    p.add_argument("--warmup-epochs", type=int, default=None)
    p.add_argument("--ema-decay", type=float, default=None)
    p.add_argument("--early-stop-patience", type=int, default=None)
    p.add_argument("--save-every", type=int, default=None)
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--resume", type=str, default=None)
    p.add_argument("--results-dir", type=str, default=None)
    args, _ = p.parse_known_args()
    return args


def get_config():
    """Merge default config with CLI overrides."""
    args = parse_args()
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    t = cfg["training"]
    if args.epochs is not None: t["epochs"] = args.epochs
    if args.batch_size is not None: t["batch_size"] = args.batch_size
    if args.lr is not None: t["lr"] = args.lr
    if args.optimizer is not None: t["optimizer"] = args.optimizer
    if args.weight_decay is not None: t["weight_decay"] = args.weight_decay
    if args.momentum is not None: t["momentum"] = args.momentum
    if args.clip_norm is not None: t["clip_norm"] = args.clip_norm
    if args.warmup_epochs is not None: t["warmup_epochs"] = args.warmup_epochs
    if args.ema_decay is not None: t["ema_decay"] = args.ema_decay
    if args.early_stop_patience is not None:
        t["early_stop_patience"] = args.early_stop_patience
    if args.save_every is not None: t["save_every"] = args.save_every
    if args.seed is not None: t["seed"] = args.seed
    if args.resume is not None: t["resume"] = args.resume
    if args.results_dir is not None: cfg["results_dir"] = args.results_dir
    return cfg


# =============================================================================
# Data Augmentation Transforms
# =============================================================================
class AugmentedTransform:
    """Train/test transform with medical-imaging-appropriate augmentations."""

    def __init__(self, train=True, cfg=None):
        self.train = train
        aug = (cfg or {}).get("augmentation", {})
        self.hflip_prob = aug.get("hflip_prob", 0.5)
        self.brightness = aug.get("brightness", 0.3)
        self.contrast = aug.get("contrast", 0.3)
        self.gamma_range = aug.get("gamma", 0.2)
        self.noise_std = aug.get("noise_std", 0.05)
        self.normalize = aug.get("normalize", True)

    def __call__(self, image, target):
        image = TF.to_tensor(image)

        if self.train:
            if torch.rand(1).item() < self.hflip_prob:
                image = TF.hflip(image)
                if len(target["boxes"]) > 0:
                    boxes = target["boxes"].clone()
                    w = image.shape[-1]
                    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                    target["boxes"] = boxes

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.brightness
                image = TF.adjust_brightness(image, factor)

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.contrast
                image = TF.adjust_contrast(image, factor)

            if torch.rand(1).item() < 0.3:
                g = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.gamma_range
                image = image.clamp(min=1e-6).pow(g)

            if torch.rand(1).item() < 0.3:
                image = (image + torch.randn_like(image) * self.noise_std).clamp(0, 1)

        if self.normalize:
            image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)

        return image, target


# =============================================================================
# Class-Frequency Weighted Sampler
# =============================================================================
def get_class_frequency_sampler(dataset):
    """Sample inversely proportional to class frequency."""
    class_counts = {1: 0, 2: 0}
    for i in range(len(dataset)):
        _, target = dataset[i]
        for lbl in target["labels"].tolist():
            if lbl in class_counts:
                class_counts[lbl] += 1

    total = sum(class_counts.values()) or 1
    class_w = {k: total / (v + 1) for k, v in class_counts.items()}

    weights = []
    for i in range(len(dataset)):
        _, target = dataset[i]
        if len(target["boxes"]) == 0:
            weights.append(1.0)
        else:
            w = max(class_w.get(int(l), 1.0) for l in target["labels"])
            weights.append(w)

    print(f"  Class counts: {class_counts}")
    print(f"  Class weights: { {k: f'{v:.2f}' for k, v in class_w.items()} }")
    return WeightedRandomSampler(weights, len(weights), replacement=True)


# =============================================================================
# Learning-Rate Scaling
# =============================================================================
def scale_lr(base_lr, batch_size, reference_bs=16):
    return base_lr * batch_size / reference_bs


# =============================================================================
# Training
# =============================================================================
def train(cfg):
    results_dir = cfg["results_dir"]
    os.makedirs(f"{results_dir}/weights", exist_ok=True)
    os.makedirs(f"{results_dir}/curves", exist_ok=True)
    os.makedirs(f"{results_dir}/explain", exist_ok=True)

    set_seed(cfg["training"]["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[FCOS] Device: {device}")

    wandb.init(
        mode=os.environ.get("WANDB_MODE", "online"), project="tbx11k", name="fcos",
        config=cfg,
    )

    train_dataset = CocoDetection(
        cfg["data"]["train_images"], cfg["data"]["train_ann"],
        transforms=AugmentedTransform(train=True, cfg=cfg),
    )
    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )

    sampler = get_class_frequency_sampler(train_dataset)
    batch_size = cfg["training"]["batch_size"]

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, sampler=sampler,
        collate_fn=collate_fn, num_workers=cfg["data"]["num_workers"],
        drop_last=True, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=cfg["data"]["num_workers"],
        pin_memory=True,
    )

    use_pretrained = False
    try:
        model = fcos_resnet50_fpn(
            weights_backbone="DEFAULT",
            num_classes=cfg["model"]["num_classes"],
        )
        use_pretrained = True
        print("  Loaded pretrained ResNet50 backbone")
    except Exception:
        import warnings
        warnings.warn("Could not load pretrained backbone, using random init")
        model = fcos_resnet50_fpn(
            weights_backbone=None,
            num_classes=cfg["model"]["num_classes"],
        )
    model.to(device)

    ema = ModelEMA(model, decay=cfg["training"]["ema_decay"])

    scaled_lr = scale_lr(cfg["training"]["lr"], batch_size)
    params = [p for p in model.parameters() if p.requires_grad]

    if cfg["training"]["optimizer"] == "AdamW":
        optimizer = torch.optim.AdamW(
            params, lr=scaled_lr,
            weight_decay=cfg["training"]["weight_decay"],
        )
    else:
        optimizer = torch.optim.SGD(
            params, lr=scaled_lr,
            momentum=cfg["training"]["momentum"],
            weight_decay=cfg["training"]["weight_decay"],
        )

    epochs = cfg["training"]["epochs"]
    warmup_epochs = cfg["training"]["warmup_epochs"]
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, total_iters=warmup_epochs,
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=1e-7,
    )
    lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, [warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs],
    )

    scaler = None
    metric_tracker = MetricTracker()

    start_epoch = 1
    best_map = 0.0
    best_epoch = -1
    patience_counter = 0

    if cfg["training"]["resume"] and os.path.exists(cfg["training"]["resume"]):
        ckpt = load_checkpoint(
            cfg["training"]["resume"], model, optimizer, scaler, lr_scheduler,
        )
        start_epoch = ckpt.get("epoch", 0) + 1
        best_map = ckpt.get("metrics", {}).get("best_map", 0.0)
        best_epoch = ckpt.get("metrics", {}).get("best_epoch", -1)
        if "ema_state_dict" in ckpt:
            ema.load_state_dict(ckpt["ema_state_dict"])
        print(
            f"[FCOS] Resumed from epoch {start_epoch}, best mAP={best_map:.4f}"
        )

    print(
        f"[FCOS] Training {epochs} epochs, LR={scaled_lr:.6f}, "
        f"warmup={warmup_epochs}, optimizer={cfg['training']['optimizer']}"
    )

    for epoch in range(start_epoch, epochs + 1):
        train_loss = train_one_epoch(
            model, optimizer, train_loader, device, epoch,
            scaler=scaler, clip_norm=cfg["training"]["clip_norm"],
            metric_tracker=metric_tracker,
        )
        lr_scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        ema.update(model)

        print(f"\n[FCOS] Validation after epoch {epoch}...")
        coco_eval = evaluate(model, val_loader, device)

        current_map = 0.0
        if coco_eval is not None:
            current_map = coco_eval.stats[0]
            val_metrics = {
                "epoch": epoch,
                "train_loss": train_loss,
                "mAP@0.5:0.95": coco_eval.stats[0],
                "mAP@0.5": coco_eval.stats[1],
                "mAP@0.75": coco_eval.stats[2],
                "AR@10": coco_eval.stats[7],
                "learning_rate": current_lr,
            }
            wandb.log(val_metrics)
            metric_tracker.update(**val_metrics)

            if current_map > best_map:
                best_map = current_map
                best_epoch = epoch
                patience_counter = 0
                save_checkpoint(
                    model, optimizer, epoch,
                    {"best_map": best_map, "best_epoch": best_epoch},
                    f"{results_dir}/weights/best_model.pth",
                    scaler=scaler, scheduler=lr_scheduler,
                    extra={"ema_state_dict": ema.state_dict()},
                )
                torch.save(
                    ema.state_dict(),
                    f"{results_dir}/weights/ema_model.pth",
                )
                print(f"  New best model! mAP={best_map:.4f} (epoch {epoch})")
            else:
                patience_counter += 1

        if epoch % cfg["training"]["save_every"] == 0:
            save_checkpoint(
                model, optimizer, epoch,
                {"train_loss": train_loss, "mAP": current_map},
                f"{results_dir}/weights/epoch_{epoch:03d}.pth",
                scaler=scaler, scheduler=lr_scheduler,
            )

        save_checkpoint(
            model, optimizer, epoch,
            {"train_loss": train_loss, "mAP": current_map},
            f"{results_dir}/weights/last_checkpoint.pth",
            scaler=scaler, scheduler=lr_scheduler,
            extra={
                "best_map": best_map,
                "best_epoch": best_epoch,
                "ema_state_dict": ema.state_dict(),
            },
        )

        if patience_counter >= cfg["training"]["early_stop_patience"]:
            print(
                f"[FCOS] Early stopping at epoch {epoch} "
                f"(no improvement for {patience_counter} epochs)"
            )
            break

        print(
            f"  LR: {current_lr:.2e} | "
            f"Patience: {patience_counter}/{cfg['training']['early_stop_patience']}"
        )
        print()

    plot_training_curves(metric_tracker, f"{results_dir}/curves")
    with open(f"{results_dir}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    print(f"\n[FCOS] Training complete. Best mAP: {best_map:.4f} at epoch {best_epoch}")
    return best_map, best_epoch


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_model(cfg):
    results_dir = cfg["results_dir"]
    print("\n[FCOS] Evaluating best model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg["training"]["batch_size"],
        shuffle=False, collate_fn=collate_fn,
        num_workers=cfg["data"]["num_workers"],
    )

    model = fcos_resnet50_fpn(
        weights_backbone="DEFAULT",
        num_classes=cfg["model"]["num_classes"],
    )
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        model.load_state_dict(
            torch.load(ema_path, map_location=device, weights_only=True)
        )
        print("  Loaded EMA weights")
    elif os.path.exists(best_path):
        model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
        print("  Loaded best model weights")
    model.to(device)
    model.eval()

    coco_eval = evaluate(
        model, val_loader, device,
        output_file=f"{results_dir}/val_preds.json",
    )

    if coco_eval is not None:
        metrics = {
            "mAP@0.5:0.95": coco_eval.stats[0],
            "mAP@0.5": coco_eval.stats[1],
            "mAP@0.75": coco_eval.stats[2],
            "mAP_small": coco_eval.stats[3],
            "mAP_medium": coco_eval.stats[4],
            "mAP_large": coco_eval.stats[5],
            "AR@1": coco_eval.stats[6],
            "AR@10": coco_eval.stats[7],
            "AR@100": coco_eval.stats[8],
        }
        with open(f"{results_dir}/metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"[FCOS] Metrics saved to {results_dir}/metrics.json")

    print("\n[FCOS] Running TTA evaluation...")
    coco_eval_tta = evaluate(
        model, val_loader, device,
        output_file=f"{results_dir}/val_preds_tta.json", tta=True,
    )
    if coco_eval_tta is not None:
        metrics_tta = {
            "mAP@0.5:0.95": coco_eval_tta.stats[0],
            "mAP@0.5": coco_eval_tta.stats[1],
            "mAP@0.75": coco_eval_tta.stats[2],
        }
        with open(f"{results_dir}/metrics_tta.json", "w") as f:
            json.dump(metrics_tta, f, indent=2)
        print(f"[FCOS] TTA metrics saved to {results_dir}/metrics_tta.json")

    confusion, class_results = compute_confusion_matrix(
        model, val_loader, device,
    )
    save_confusion_matrix_plot(
        confusion, f"{results_dir}/confusion_matrix.png",
    )

    test_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    if os.path.isfile(test_ann):
        test_dataset = CocoDetection(
            cfg["data"].get("test_images", "dataset/coco/test"),
            test_ann,
            transforms=AugmentedTransform(train=False, cfg=cfg),
        )
        test_loader = DataLoader(
            test_dataset, batch_size=cfg["training"]["batch_size"],
            shuffle=False, collate_fn=collate_fn,
            num_workers=cfg["data"]["num_workers"],
        )
        evaluate_test(
            model, test_loader, device,
            f"{results_dir}/test_preds.json",
            model_name=cfg.get("model", {}).get("name", "FCOS-ResNet50-FPN"),
        )
    else:
        print("[TEST] No annotations found. Running inference-only prediction.")
        test_images = cfg["data"].get("test_images", "dataset/coco/test")
        if os.path.isdir(test_images):
            test_dataset = CocoDetection(
                test_images, test_images,
                transforms=AugmentedTransform(train=False, cfg=cfg),
            )
            test_loader = DataLoader(
                test_dataset, batch_size=cfg["training"]["batch_size"],
                shuffle=False, collate_fn=collate_fn,
                num_workers=cfg["data"]["num_workers"],
            )
            evaluate_test(
                model, test_loader, device,
                f"{results_dir}/test_preds.json",
                model_name=cfg.get("model", {}).get("name", "FCOS-ResNet50-FPN"),
            )


# =============================================================================
# XAI — Grad-CAM with TP / FP / FN Analysis
# =============================================================================
def run_xai(cfg):
    results_dir = cfg["results_dir"]
    print("\n[FCOS] Generating XAI explanations...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = fcos_resnet50_fpn(
        weights_backbone="DEFAULT",
        num_classes=cfg["model"]["num_classes"],
    )
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        model.load_state_dict(
            torch.load(ema_path, map_location=device, weights_only=True)
        )
    elif os.path.exists(best_path):
        model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
    model.to(device)
    model.eval()

    xai_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    xai_img_dir = cfg["data"].get("test_images", "dataset/coco/test")
    if not os.path.isfile(xai_ann):
        xai_ann = cfg["data"].get("val_ann", "dataset/coco/val.json")
        xai_img_dir = cfg["data"]["val_images"]
    if not os.path.exists(xai_ann):
        print("[FCOS] No dataset available for XAI.")
        return

    xai_dataset = CocoDetection(
        xai_img_dir, xai_ann,
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    loader = DataLoader(
        xai_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_fn, num_workers=2,
    )

    target_layer = get_target_layer(model, "fcos")
    gradcam = GradCAM(model, target_layer)

    from torchvision.ops import box_iou as _box_iou

    categories = {1: "ActiveTB", 2: "ObsoleteTB"}
    xai_counts = {c: {"tp": 0, "fp": 0, "fn": 0} for c in categories}
    max_per_type = 3

    for images, targets in loader:
        if all(
            sum(v.values()) >= max_per_type * 3 for v in xai_counts.values()
        ):
            break

        image = images[0].unsqueeze(0).to(device)
        gt_boxes = targets[0]["boxes"].numpy()
        gt_labels = targets[0]["labels"].numpy()

        with torch.no_grad():
            outputs = model(image)
        pred_boxes = outputs[0]["boxes"].cpu().numpy()
        pred_scores = outputs[0]["scores"].cpu().numpy()
        pred_labels = outputs[0]["labels"].cpu().numpy()

        pred_type = ["FP"] * len(pred_boxes)
        matched_gt = set()
        if len(gt_boxes) > 0 and len(pred_boxes) > 0:
            iou_mat = _box_iou(
                torch.tensor(gt_boxes), torch.tensor(pred_boxes)
            ).numpy()
            for gi in range(len(gt_boxes)):
                best_pi, best_iou = -1, 0.5
                for pi in range(len(pred_boxes)):
                    if iou_mat[gi, pi] > best_iou:
                        best_iou = iou_mat[gi, pi]
                        best_pi = pi
                if best_pi >= 0 and pred_labels[best_pi] == gt_labels[gi]:
                    pred_type[best_pi] = "TP"
                    matched_gt.add(gi)

        for pi, ptype in enumerate(pred_type):
            if pi >= len(pred_scores) or pred_scores[pi] < 0.3:
                continue
            cat = int(pred_labels[pi])
            if cat not in categories:
                continue
            if xai_counts[cat][ptype.lower()] >= max_per_type:
                continue

            with torch.set_grad_enabled(True):
                heatmap = gradcam.generate(image, target_class=cat)
            if heatmap is None:
                continue

            overlay = overlay_heatmap(image, heatmap)
            det = {
                "boxes": pred_boxes[pi : pi + 1],
                "scores": pred_scores[pi : pi + 1],
                "labels": pred_labels[pi : pi + 1],
            }
            overlay = draw_detections(overlay, det)
            name = categories[cat]
            n = xai_counts[cat][ptype.lower()]
            overlay.save(f"{results_dir}/explain/{name}_{ptype}_{n}.png")
            xai_counts[cat][ptype.lower()] += 1

        for gi in range(len(gt_boxes)):
            if gi in matched_gt:
                continue
            cat = int(gt_labels[gi])
            if cat not in categories:
                continue
            if xai_counts[cat]["fn"] >= max_per_type:
                continue

            with torch.set_grad_enabled(True):
                heatmap = gradcam.generate(image, target_class=cat)
            if heatmap is None:
                continue

            overlay = overlay_heatmap(image, heatmap)
            det = {
                "boxes": gt_boxes[gi : gi + 1],
                "scores": np.array([0.0]),
                "labels": gt_labels[gi : gi + 1],
            }
            overlay = draw_detections(overlay, det)
            name = categories[cat]
            n = xai_counts[cat]["fn"]
            overlay.save(f"{results_dir}/explain/{name}_FN_{n}.png")
            xai_counts[cat]["fn"] += 1

    gradcam.remove_hooks()
    print("[FCOS] XAI complete.")
    for cat_id, counts in xai_counts.items():
        print(
            f"  {categories[cat_id]}: "
            f"TP={counts['tp']} FP={counts['fp']} FN={counts['fn']}"
        )


# =============================================================================
# Main
# =============================================================================
def main():
    cfg = get_config()
    train(cfg)
    evaluate_model(cfg)
    run_xai(cfg)


if __name__ == "__main__":
    main()


### 12. Write `train_retinanet.py`

In [ ]:
%%writefile train_retinanet.py
#!/usr/bin/env python3
"""RetinaNet Training Pipeline — Research-Grade Implementation for TBX11K.

Usage:
    python train_retinanet.py
    python train_retinanet.py --epochs 50 --lr 0.001
    python train_retinanet.py --resume results/retinanet/weights/last_checkpoint.pth
    python train_retinanet.py --optimizer AdamW --batch-size 4
"""

import os
import sys
import json
import copy
import argparse

import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.anchor_utils import AnchorGenerator
import torchvision.transforms as T
import torchvision.transforms.functional as TF

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass
from utils.coco_dataset import CocoDetection, collate_fn
from utils.engine import (
    set_seed, MetricTracker, save_checkpoint, load_checkpoint,
    train_one_epoch, evaluate, evaluate_test,
    compute_confusion_matrix, save_confusion_matrix_plot,
    plot_training_curves,
)
from utils.ema import ModelEMA
from explain.gradcam import GradCAM, get_target_layer
from explain.visualize import overlay_heatmap, draw_detections

CLASS_NAMES = ["Background", "ActiveTuberculosis", "ObsoletePulmonaryTuberculosis"]

# =============================================================================
# Configuration
# =============================================================================
DEFAULT_CONFIG = {
    "model": {
        "name": "RetinaNet-ResNet50-FPN-V2",
        "num_classes": len(CLASS_NAMES),
        "use_custom_anchors": False,
        "anchor_sizes": ((16,), (32,), (64,), (128,), (256,)),
        "aspect_ratios": ((0.5, 1.0, 2.0),) * 5,
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 0.005,
        "optimizer": "SGD",
        "weight_decay": 1e-4,
        "momentum": 0.9,
        "clip_norm": 1.0,
        "warmup_epochs": 5,
        "ema_decay": 0.99,
        "early_stop_patience": 15,
        "save_every": 10,
        "seed": 42,
        "resume": None,
    },
    "augmentation": {
        "hflip_prob": 0.5,
        "brightness": 0.3,
        "contrast": 0.3,
        "gamma": 0.2,
        "noise_std": 0.05,
        "normalize": True,
    },
    "data": {
        "train_images": "dataset/coco/train",
        "val_images": "dataset/coco/val",
        "train_ann": "dataset/coco/train.json",
        "val_ann": "dataset/coco/val.json",
        "test_images": "dataset/coco/test",
        "test_ann": "dataset/coco/test.json",
        "num_workers": 2,
    },
    "results_dir": "results/retinanet",
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# =============================================================================
# CLI Argument Parser
# =============================================================================
def parse_args():
    p = argparse.ArgumentParser(description="RetinaNet TB Detection Training")
    p.add_argument("--epochs", type=int, default=None)
    p.add_argument("--batch-size", type=int, default=None)
    p.add_argument("--lr", type=float, default=None)
    p.add_argument("--optimizer", type=str, default=None, choices=["SGD", "AdamW"])
    p.add_argument("--weight-decay", type=float, default=None)
    p.add_argument("--momentum", type=float, default=None)
    p.add_argument("--clip-norm", type=float, default=None)
    p.add_argument("--warmup-epochs", type=int, default=None)
    p.add_argument("--ema-decay", type=float, default=None)
    p.add_argument("--early-stop-patience", type=int, default=None)
    p.add_argument("--save-every", type=int, default=None)
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--resume", type=str, default=None)
    p.add_argument("--results-dir", type=str, default=None)
    p.add_argument("--no-custom-anchors", action="store_true",
                   help="Use default RetinaNet anchors instead of small-lesion anchors")
    args, _ = p.parse_known_args()
    return args


def get_config():
    args = parse_args()
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    t = cfg["training"]
    m = cfg["model"]
    if args.epochs is not None: t["epochs"] = args.epochs
    if args.batch_size is not None: t["batch_size"] = args.batch_size
    if args.lr is not None: t["lr"] = args.lr
    if args.optimizer is not None: t["optimizer"] = args.optimizer
    if args.weight_decay is not None: t["weight_decay"] = args.weight_decay
    if args.momentum is not None: t["momentum"] = args.momentum
    if args.clip_norm is not None: t["clip_norm"] = args.clip_norm
    if args.warmup_epochs is not None: t["warmup_epochs"] = args.warmup_epochs
    if args.ema_decay is not None: t["ema_decay"] = args.ema_decay
    if args.early_stop_patience is not None:
        t["early_stop_patience"] = args.early_stop_patience
    if args.save_every is not None: t["save_every"] = args.save_every
    if args.seed is not None: t["seed"] = args.seed
    if args.resume is not None: t["resume"] = args.resume
    if args.results_dir is not None: cfg["results_dir"] = args.results_dir
    if args.no_custom_anchors: m["use_custom_anchors"] = False
    return cfg


# =============================================================================
# Data Augmentation Transforms
# =============================================================================
class AugmentedTransform:
    def __init__(self, train=True, cfg=None):
        self.train = train
        aug = (cfg or {}).get("augmentation", {})
        self.hflip_prob = aug.get("hflip_prob", 0.5)
        self.brightness = aug.get("brightness", 0.3)
        self.contrast = aug.get("contrast", 0.3)
        self.gamma_range = aug.get("gamma", 0.2)
        self.noise_std = aug.get("noise_std", 0.05)
        self.normalize = aug.get("normalize", True)

    def __call__(self, image, target):
        image = TF.to_tensor(image)

        if self.train:
            if torch.rand(1).item() < self.hflip_prob:
                image = TF.hflip(image)
                if len(target["boxes"]) > 0:
                    boxes = target["boxes"].clone()
                    w = image.shape[-1]
                    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                    target["boxes"] = boxes

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.brightness
                image = TF.adjust_brightness(image, factor)

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.contrast
                image = TF.adjust_contrast(image, factor)

            if torch.rand(1).item() < 0.3:
                g = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.gamma_range
                image = image.clamp(min=1e-6).pow(g)

            if torch.rand(1).item() < 0.3:
                image = (image + torch.randn_like(image) * self.noise_std).clamp(0, 1)

        if self.normalize:
            image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)

        return image, target


# =============================================================================
# Class-Frequency Weighted Sampler
# =============================================================================
def get_class_frequency_sampler(dataset):
    class_counts = {1: 0, 2: 0}
    for i in range(len(dataset)):
        _, target = dataset[i]
        for lbl in target["labels"].tolist():
            if lbl in class_counts:
                class_counts[lbl] += 1

    total = sum(class_counts.values()) or 1
    class_w = {k: total / (v + 1) for k, v in class_counts.items()}

    weights = []
    for i in range(len(dataset)):
        _, target = dataset[i]
        if len(target["boxes"]) == 0:
            weights.append(1.0)
        else:
            w = max(class_w.get(int(l), 1.0) for l in target["labels"])
            weights.append(w)

    print(f"  Class counts: {class_counts}")
    print(f"  Class weights: { {k: f'{v:.2f}' for k, v in class_w.items()} }")
    return WeightedRandomSampler(weights, len(weights), replacement=True)


# =============================================================================
# Learning-Rate Scaling
# =============================================================================
def scale_lr(base_lr, batch_size, reference_bs=16):
    return base_lr * batch_size / reference_bs


# =============================================================================
# Build RetinaNet with optional custom anchors for small lesions
# =============================================================================
def build_retinanet(cfg):
    num_classes = cfg["model"]["num_classes"]
    use_custom = cfg["model"].get("use_custom_anchors", True)
    use_pretrained = False

    try:
        model = retinanet_resnet50_fpn_v2(
            weights="DEFAULT", num_classes=num_classes,
        )
        use_pretrained = True
        print("  Loaded COCO-pretrained RetinaNet-V2 (backbone + neck)")
    except Exception:
        try:
            model = retinanet_resnet50_fpn_v2(
                weights_backbone="DEFAULT", num_classes=num_classes,
            )
            use_pretrained = True
            print("  Loaded backbone-only pretrained weights")
        except Exception:
            import warnings
            warnings.warn("Could not load pretrained weights, using random init. Disabling AMP.")
            model = retinanet_resnet50_fpn_v2(
                weights_backbone=None, num_classes=num_classes,
            )

    if use_custom:
        anchor_sizes = cfg["model"]["anchor_sizes"]
        aspect_ratios = cfg["model"]["aspect_ratios"]
        model.anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)
        print(f"  Custom anchors: sizes={anchor_sizes}, ratios={aspect_ratios[0]}")
        n_anchors = sum(len(s) * len(aspect_ratios[0]) for s in anchor_sizes)
        print(f"  Total anchors per location: {n_anchors}")

    return model, use_pretrained


# =============================================================================
# Training
# =============================================================================
def train(cfg):
    results_dir = cfg["results_dir"]
    os.makedirs(f"{results_dir}/weights", exist_ok=True)
    os.makedirs(f"{results_dir}/curves", exist_ok=True)
    os.makedirs(f"{results_dir}/explain", exist_ok=True)

    set_seed(cfg["training"]["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[RetinaNet] Device: {device}")

    wandb.init(
        mode=os.environ.get("WANDB_MODE", "online"), project="tbx11k", name="retinanet",
        config=cfg,
    )

    train_dataset = CocoDetection(
        cfg["data"]["train_images"], cfg["data"]["train_ann"],
        transforms=AugmentedTransform(train=True, cfg=cfg),
    )
    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )

    sampler = get_class_frequency_sampler(train_dataset)
    batch_size = cfg["training"]["batch_size"]

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, sampler=sampler,
        collate_fn=collate_fn, num_workers=cfg["data"]["num_workers"],
        drop_last=True, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=cfg["data"]["num_workers"],
        pin_memory=True,
    )

    model, use_pretrained = build_retinanet(cfg)
    model.to(device)

    ema = ModelEMA(model, decay=cfg["training"]["ema_decay"])

    scaled_lr = scale_lr(cfg["training"]["lr"], batch_size)
    params = [p for p in model.parameters() if p.requires_grad]

    if cfg["training"]["optimizer"] == "AdamW":
        optimizer = torch.optim.AdamW(
            params, lr=scaled_lr,
            weight_decay=cfg["training"]["weight_decay"],
        )
    else:
        optimizer = torch.optim.SGD(
            params, lr=scaled_lr,
            momentum=cfg["training"]["momentum"],
            weight_decay=cfg["training"]["weight_decay"],
        )

    epochs = cfg["training"]["epochs"]
    warmup_epochs = cfg["training"]["warmup_epochs"]
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, total_iters=warmup_epochs,
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=1e-7,
    )
    lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, [warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs],
    )

    scaler = None
    metric_tracker = MetricTracker()

    start_epoch = 1
    best_map = 0.0
    best_epoch = -1
    patience_counter = 0

    if cfg["training"]["resume"] and os.path.exists(cfg["training"]["resume"]):
        ckpt = load_checkpoint(
            cfg["training"]["resume"], model, optimizer, scaler, lr_scheduler,
        )
        start_epoch = ckpt.get("epoch", 0) + 1
        best_map = ckpt.get("metrics", {}).get("best_map", 0.0)
        best_epoch = ckpt.get("metrics", {}).get("best_epoch", -1)
        if "ema_state_dict" in ckpt:
            ema.load_state_dict(ckpt["ema_state_dict"])
        print(
            f"[RetinaNet] Resumed from epoch {start_epoch}, "
            f"best mAP={best_map:.4f}"
        )

    print(
        f"[RetinaNet] Training {epochs} epochs, LR={scaled_lr:.6f}, "
        f"warmup={warmup_epochs}, optimizer={cfg['training']['optimizer']}"
    )

    for epoch in range(start_epoch, epochs + 1):
        train_loss = train_one_epoch(
            model, optimizer, train_loader, device, epoch,
            scaler=scaler, clip_norm=cfg["training"]["clip_norm"],
            metric_tracker=metric_tracker,
        )
        lr_scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        ema.update(model)

        print(f"\n[RetinaNet] Validation after epoch {epoch}...")
        coco_eval = evaluate(model, val_loader, device)

        current_map = 0.0
        if coco_eval is not None:
            current_map = coco_eval.stats[0]
            val_metrics = {
                "epoch": epoch,
                "train_loss": train_loss,
                "mAP@0.5:0.95": coco_eval.stats[0],
                "mAP@0.5": coco_eval.stats[1],
                "mAP@0.75": coco_eval.stats[2],
                "AR@10": coco_eval.stats[7],
                "learning_rate": current_lr,
            }
            wandb.log(val_metrics)
            metric_tracker.update(**val_metrics)

            if current_map > best_map:
                best_map = current_map
                best_epoch = epoch
                patience_counter = 0
                save_checkpoint(
                    model, optimizer, epoch,
                    {"best_map": best_map, "best_epoch": best_epoch},
                    f"{results_dir}/weights/best_model.pth",
                    scaler=scaler, scheduler=lr_scheduler,
                    extra={"ema_state_dict": ema.state_dict()},
                )
                torch.save(
                    ema.state_dict(),
                    f"{results_dir}/weights/ema_model.pth",
                )
                print(
                    f"  New best model! mAP={best_map:.4f} (epoch {epoch})"
                )
            else:
                patience_counter += 1

        if epoch % cfg["training"]["save_every"] == 0:
            save_checkpoint(
                model, optimizer, epoch,
                {"train_loss": train_loss, "mAP": current_map},
                f"{results_dir}/weights/epoch_{epoch:03d}.pth",
                scaler=scaler, scheduler=lr_scheduler,
            )

        save_checkpoint(
            model, optimizer, epoch,
            {"train_loss": train_loss, "mAP": current_map},
            f"{results_dir}/weights/last_checkpoint.pth",
            scaler=scaler, scheduler=lr_scheduler,
            extra={
                "best_map": best_map,
                "best_epoch": best_epoch,
                "ema_state_dict": ema.state_dict(),
            },
        )

        if patience_counter >= cfg["training"]["early_stop_patience"]:
            print(
                f"[RetinaNet] Early stopping at epoch {epoch} "
                f"(no improvement for {patience_counter} epochs)"
            )
            break

        print(
            f"  LR: {current_lr:.2e} | "
            f"Patience: {patience_counter}/{cfg['training']['early_stop_patience']}"
        )
        print()

    plot_training_curves(metric_tracker, f"{results_dir}/curves")
    with open(f"{results_dir}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    print(
        f"\n[RetinaNet] Training complete. "
        f"Best mAP: {best_map:.4f} at epoch {best_epoch}"
    )
    return best_map, best_epoch


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_model(cfg):
    results_dir = cfg["results_dir"]
    print("\n[RetinaNet] Evaluating best model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg["training"]["batch_size"],
        shuffle=False, collate_fn=collate_fn,
        num_workers=cfg["data"]["num_workers"],
    )

    model = build_retinanet(cfg)
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        model.load_state_dict(
            torch.load(ema_path, map_location=device, weights_only=True)
        )
        print("  Loaded EMA weights")
    elif os.path.exists(best_path):
        model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
        print("  Loaded best model weights")
    model.to(device)
    model.eval()

    coco_eval = evaluate(
        model, val_loader, device,
        output_file=f"{results_dir}/val_preds.json",
    )

    if coco_eval is not None:
        metrics = {
            "mAP@0.5:0.95": coco_eval.stats[0],
            "mAP@0.5": coco_eval.stats[1],
            "mAP@0.75": coco_eval.stats[2],
            "mAP_small": coco_eval.stats[3],
            "mAP_medium": coco_eval.stats[4],
            "mAP_large": coco_eval.stats[5],
            "AR@1": coco_eval.stats[6],
            "AR@10": coco_eval.stats[7],
            "AR@100": coco_eval.stats[8],
        }
        with open(f"{results_dir}/metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"[RetinaNet] Metrics saved to {results_dir}/metrics.json")

    print("\n[RetinaNet] Running TTA evaluation...")
    coco_eval_tta = evaluate(
        model, val_loader, device,
        output_file=f"{results_dir}/val_preds_tta.json", tta=True,
    )
    if coco_eval_tta is not None:
        metrics_tta = {
            "mAP@0.5:0.95": coco_eval_tta.stats[0],
            "mAP@0.5": coco_eval_tta.stats[1],
            "mAP@0.75": coco_eval_tta.stats[2],
        }
        with open(f"{results_dir}/metrics_tta.json", "w") as f:
            json.dump(metrics_tta, f, indent=2)
        print(
            f"[RetinaNet] TTA metrics saved to {results_dir}/metrics_tta.json"
        )

    confusion, class_results = compute_confusion_matrix(
        model, val_loader, device,
    )
    save_confusion_matrix_plot(
        confusion, f"{results_dir}/confusion_matrix.png",
    )

    test_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    if os.path.isfile(test_ann):
        test_dataset = CocoDetection(
            cfg["data"].get("test_images", "dataset/coco/test"),
            test_ann,
            transforms=AugmentedTransform(train=False, cfg=cfg),
        )
        test_loader = DataLoader(
            test_dataset, batch_size=cfg["training"]["batch_size"],
            shuffle=False, collate_fn=collate_fn,
            num_workers=cfg["data"]["num_workers"],
        )
        evaluate_test(
            model, test_loader, device,
            f"{results_dir}/test_preds.json",
            model_name=cfg.get("model", {}).get("name", "RetinaNet-ResNet50-FPN"),
        )
    else:
        print("[TEST] No annotations found. Running inference-only prediction.")


# =============================================================================
# XAI — Grad-CAM with TP / FP / FN Analysis
# =============================================================================
def run_xai(cfg):
    results_dir = cfg["results_dir"]
    print("\n[RetinaNet] Generating XAI explanations...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_retinanet(cfg)
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        model.load_state_dict(
            torch.load(ema_path, map_location=device, weights_only=True)
        )
    elif os.path.exists(best_path):
        model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
    model.to(device)
    model.eval()

    xai_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    xai_img_dir = cfg["data"].get("test_images", "dataset/coco/test")
    if not os.path.isfile(xai_ann):
        xai_ann = cfg["data"].get("val_ann", "dataset/coco/val.json")
        xai_img_dir = cfg["data"]["val_images"]
    if not os.path.exists(xai_ann):
        print("[RetinaNet] No dataset available for XAI.")
        return

    xai_dataset = CocoDetection(
        xai_img_dir, xai_ann,
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    loader = DataLoader(
        xai_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_fn, num_workers=2,
    )

    target_layer = get_target_layer(model, "retinanet")
    gradcam = GradCAM(model, target_layer)

    from torchvision.ops import box_iou as _box_iou

    categories = {1: "ActiveTB", 2: "ObsoleteTB"}
    xai_counts = {c: {"tp": 0, "fp": 0, "fn": 0} for c in categories}
    max_per_type = 3

    for images, targets in loader:
        if all(
            sum(v.values()) >= max_per_type * 3 for v in xai_counts.values()
        ):
            break

        image = images[0].unsqueeze(0).to(device)
        gt_boxes = targets[0]["boxes"].numpy()
        gt_labels = targets[0]["labels"].numpy()

        with torch.no_grad():
            outputs = model(image)
        pred_boxes = outputs[0]["boxes"].cpu().numpy()
        pred_scores = outputs[0]["scores"].cpu().numpy()
        pred_labels = outputs[0]["labels"].cpu().numpy()

        pred_type = ["FP"] * len(pred_boxes)
        matched_gt = set()
        if len(gt_boxes) > 0 and len(pred_boxes) > 0:
            iou_mat = _box_iou(
                torch.tensor(gt_boxes), torch.tensor(pred_boxes)
            ).numpy()
            for gi in range(len(gt_boxes)):
                best_pi, best_iou = -1, 0.5
                for pi in range(len(pred_boxes)):
                    if iou_mat[gi, pi] > best_iou:
                        best_iou = iou_mat[gi, pi]
                        best_pi = pi
                if best_pi >= 0 and pred_labels[best_pi] == gt_labels[gi]:
                    pred_type[best_pi] = "TP"
                    matched_gt.add(gi)

        for pi, ptype in enumerate(pred_type):
            if pi >= len(pred_scores) or pred_scores[pi] < 0.3:
                continue
            cat = int(pred_labels[pi])
            if cat not in categories:
                continue
            if xai_counts[cat][ptype.lower()] >= max_per_type:
                continue

            with torch.set_grad_enabled(True):
                heatmap = gradcam.generate(image, target_class=cat)
            if heatmap is None:
                continue

            overlay = overlay_heatmap(image, heatmap)
            det = {
                "boxes": pred_boxes[pi : pi + 1],
                "scores": pred_scores[pi : pi + 1],
                "labels": pred_labels[pi : pi + 1],
            }
            overlay = draw_detections(overlay, det)
            name = categories[cat]
            n = xai_counts[cat][ptype.lower()]
            overlay.save(f"{results_dir}/explain/{name}_{ptype}_{n}.png")
            xai_counts[cat][ptype.lower()] += 1

        for gi in range(len(gt_boxes)):
            if gi in matched_gt:
                continue
            cat = int(gt_labels[gi])
            if cat not in categories:
                continue
            if xai_counts[cat]["fn"] >= max_per_type:
                continue

            with torch.set_grad_enabled(True):
                heatmap = gradcam.generate(image, target_class=cat)
            if heatmap is None:
                continue

            overlay = overlay_heatmap(image, heatmap)
            det = {
                "boxes": gt_boxes[gi : gi + 1],
                "scores": np.array([0.0]),
                "labels": gt_labels[gi : gi + 1],
            }
            overlay = draw_detections(overlay, det)
            name = categories[cat]
            n = xai_counts[cat]["fn"]
            overlay.save(f"{results_dir}/explain/{name}_FN_{n}.png")
            xai_counts[cat]["fn"] += 1

    gradcam.remove_hooks()
    print("[RetinaNet] XAI complete.")
    for cat_id, counts in xai_counts.items():
        print(
            f"  {categories[cat_id]}: "
            f"TP={counts['tp']} FP={counts['fp']} FN={counts['fn']}"
        )


# =============================================================================
# Main
# =============================================================================
def main():
    cfg = get_config()
    train(cfg)
    evaluate_model(cfg)
    run_xai(cfg)


if __name__ == "__main__":
    main()


### 13. Write `train_detr.py`

In [ ]:
%%writefile train_detr.py
#!/usr/bin/env python3
"""DETR Training Pipeline — Research-Grade Implementation for TBX11K.

Usage:
    python train_detr.py
    python train_detr.py --epochs 200 --lr 1e-4
    python train_detr.py --resume results/detr/weights/last_checkpoint.pth
    python train_detr.py --grad-accum 4
"""

import os
import sys
import json
import copy
import argparse

import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import DataLoader
from torchvision.models.detection import detr_resnet50
import torchvision.transforms as T
import torchvision.transforms.functional as TF

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass
from utils.coco_dataset import CocoDetection, collate_fn
from utils.engine import (
    set_seed, MetricTracker, save_checkpoint, load_checkpoint,
    train_one_epoch, evaluate, evaluate_test,
    compute_confusion_matrix, save_confusion_matrix_plot,
    plot_training_curves,
)
from utils.ema import ModelEMA
from explain.detr_attention import DETRAttentionExtractor
from explain.visualize import overlay_heatmap, draw_detections

CLASS_NAMES = {0: "Background", 1: "ActiveTuberculosis", 2: "ObsoletePulmonaryTuberculosis"}
CLASS_COLORS = {1: (255, 0, 0), 2: (0, 200, 255)}
NUM_CLASSES = 3

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# =============================================================================
# Configuration
# =============================================================================
DEFAULT_CONFIG = {
    "model": {
        "name": "DETR-ResNet50",
        "num_classes": NUM_CLASSES,
    },
    "training": {
        "epochs": 150,
        "batch_size": 4,
        "lr_backbone": 1e-5,
        "lr_transformer": 1e-4,
        "lr_head": 1e-4,
        "weight_decay": 1e-4,
        "clip_norm": 0.1,
        "warmup_epochs": 10,
        "ema_decay": 0.99,
        "early_stop_patience": 30,
        "save_every": 10,
        "seed": 42,
        "resume": None,
        "gradient_accumulation_steps": 4,
    },
    "augmentation": {
        "hflip_prob": 0.5,
        "brightness": 0.3,
        "contrast": 0.3,
        "gamma": 0.2,
        "noise_std": 0.05,
        "normalize": True,
    },
    "data": {
        "train_images": "dataset/coco/train",
        "val_images": "dataset/coco/val",
        "train_ann": "dataset/coco/train.json",
        "val_ann": "dataset/coco/val.json",
        "test_images": "dataset/coco/test",
        "test_ann": "dataset/coco/test.json",
        "num_workers": 2,
    },
    "results_dir": "results/detr",
}


# =============================================================================
# CLI Argument Parser
# =============================================================================
def parse_args():
    p = argparse.ArgumentParser(description="DETR TB Detection Training")
    p.add_argument("--epochs", type=int, default=None)
    p.add_argument("--batch-size", type=int, default=None)
    p.add_argument("--lr-backbone", type=float, default=None)
    p.add_argument("--lr-transformer", type=float, default=None)
    p.add_argument("--lr-head", type=float, default=None)
    p.add_argument("--weight-decay", type=float, default=None)
    p.add_argument("--clip-norm", type=float, default=None)
    p.add_argument("--warmup-epochs", type=int, default=None)
    p.add_argument("--ema-decay", type=float, default=None)
    p.add_argument("--early-stop-patience", type=int, default=None)
    p.add_argument("--save-every", type=int, default=None)
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--resume", type=str, default=None)
    p.add_argument("--results-dir", type=str, default=None)
    p.add_argument("--grad-accum", type=int, default=None,
                   help="Gradient accumulation steps (default: 4, effective batch=16)")
    args, _ = p.parse_known_args()
    return args


def get_config():
    args = parse_args()
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    t = cfg["training"]
    if args.epochs is not None: t["epochs"] = args.epochs
    if args.batch_size is not None: t["batch_size"] = args.batch_size
    if args.lr_backbone is not None: t["lr_backbone"] = args.lr_backbone
    if args.lr_transformer is not None: t["lr_transformer"] = args.lr_transformer
    if args.lr_head is not None: t["lr_head"] = args.lr_head
    if args.weight_decay is not None: t["weight_decay"] = args.weight_decay
    if args.clip_norm is not None: t["clip_norm"] = args.clip_norm
    if args.warmup_epochs is not None: t["warmup_epochs"] = args.warmup_epochs
    if args.ema_decay is not None: t["ema_decay"] = args.ema_decay
    if args.early_stop_patience is not None:
        t["early_stop_patience"] = args.early_stop_patience
    if args.save_every is not None: t["save_every"] = args.save_every
    if args.seed is not None: t["seed"] = args.seed
    if args.resume is not None: t["resume"] = args.resume
    if args.results_dir is not None: cfg["results_dir"] = args.results_dir
    if args.grad_accum is not None: t["gradient_accumulation_steps"] = args.grad_accum
    return cfg


# =============================================================================
# Data Augmentation Transforms
# =============================================================================
class AugmentedTransform:
    def __init__(self, train=True, cfg=None):
        self.train = train
        aug = (cfg or {}).get("augmentation", {})
        self.hflip_prob = aug.get("hflip_prob", 0.5)
        self.brightness = aug.get("brightness", 0.3)
        self.contrast = aug.get("contrast", 0.3)
        self.gamma_range = aug.get("gamma", 0.2)
        self.noise_std = aug.get("noise_std", 0.05)
        self.normalize = aug.get("normalize", True)

    def __call__(self, image, target):
        image = TF.to_tensor(image)

        if self.train:
            if torch.rand(1).item() < self.hflip_prob:
                image = TF.hflip(image)
                if len(target["boxes"]) > 0:
                    boxes = target["boxes"].clone()
                    w = image.shape[-1]
                    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                    target["boxes"] = boxes

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.brightness
                image = TF.adjust_brightness(image, factor)

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.contrast
                image = TF.adjust_contrast(image, factor)

            if torch.rand(1).item() < 0.3:
                g = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.gamma_range
                image = image.clamp(min=1e-6).pow(g)

            if torch.rand(1).item() < 0.3:
                image = (image + torch.randn_like(image) * self.noise_std).clamp(0, 1)

        if self.normalize:
            image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)

        return image, target


# =============================================================================
# Build DETR with COCO-pretrained weights
# =============================================================================
def build_detr(cfg):
    num_classes = cfg["model"]["num_classes"]
    use_pretrained = False

    try:
        model = detr_resnet50(
            weights="DEFAULT", num_classes=num_classes,
        )
        use_pretrained = True
        print("  Loaded COCO-pretrained DETR (backbone + transformer + head)")
    except Exception:
        try:
            model = detr_resnet50(
                weights_backbone="DEFAULT", num_classes=num_classes,
            )
            use_pretrained = True
            print("  Loaded backbone-only pretrained weights")
        except Exception:
            import warnings
            warnings.warn("Could not load pretrained weights, using random init. Disabling AMP.")
            model = detr_resnet50(
                weights_backbone=None, num_classes=num_classes,
            )

    return model, use_pretrained


# =============================================================================
# Separate parameter groups: backbone vs transformer vs head
# =============================================================================
def get_param_groups(model, cfg):
    backbone_params = []
    transformer_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(param)
        elif "transformer" in name:
            transformer_params.append(param)
        else:
            head_params.append(param)

    lr_bb = cfg["training"]["lr_backbone"]
    lr_tr = cfg["training"]["lr_transformer"]
    lr_hd = cfg["training"]["lr_head"]
    wd = cfg["training"]["weight_decay"]

    param_groups = [
        {"params": backbone_params, "lr": lr_bb, "name": "backbone"},
        {"params": transformer_params, "lr": lr_tr, "name": "transformer"},
        {"params": head_params, "lr": lr_hd, "name": "head"},
    ]

    print(f"  Param groups: backbone={len(backbone_params)} ({lr_bb:.1e}), "
          f"transformer={len(transformer_params)} ({lr_tr:.1e}), "
          f"head={len(head_params)} ({lr_hd:.1e})")

    return param_groups


# =============================================================================
# Training
# =============================================================================
def train(cfg):
    results_dir = cfg["results_dir"]
    os.makedirs(f"{results_dir}/weights", exist_ok=True)
    os.makedirs(f"{results_dir}/curves", exist_ok=True)
    os.makedirs(f"{results_dir}/explain", exist_ok=True)

    set_seed(cfg["training"]["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[DETR] Device: {device}")

    wandb.init(
        mode=os.environ.get("WANDB_MODE", "online"), project="tbx11k", name="detr",
        config=cfg,
    )

    train_dataset = CocoDetection(
        cfg["data"]["train_images"], cfg["data"]["train_ann"],
        transforms=AugmentedTransform(train=True, cfg=cfg),
    )
    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )

    batch_size = cfg["training"]["batch_size"]

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=cfg["data"]["num_workers"],
        drop_last=True, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=cfg["data"]["num_workers"],
        pin_memory=True,
    )

    model, use_pretrained = build_detr(cfg)
    model.to(device)

    ema = ModelEMA(model, decay=cfg["training"]["ema_decay"])

    param_groups = get_param_groups(model, cfg)
    optimizer = torch.optim.AdamW(
        param_groups, weight_decay=cfg["training"]["weight_decay"],
    )

    epochs = cfg["training"]["epochs"]
    warmup_epochs = cfg["training"]["warmup_epochs"]
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, total_iters=warmup_epochs,
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=1e-8,
    )
    lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, [warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs],
    )

    scaler = None
    metric_tracker = MetricTracker()
    grad_accum = cfg["training"]["gradient_accumulation_steps"]

    start_epoch = 1
    best_map = 0.0
    best_epoch = -1
    patience_counter = 0

    if cfg["training"]["resume"] and os.path.exists(cfg["training"]["resume"]):
        ckpt = load_checkpoint(
            cfg["training"]["resume"], model, optimizer, scaler, lr_scheduler,
        )
        start_epoch = ckpt.get("epoch", 0) + 1
        best_map = ckpt.get("metrics", {}).get("best_map", 0.0)
        best_epoch = ckpt.get("metrics", {}).get("best_epoch", -1)
        if "ema_state_dict" in ckpt:
            ema.load_state_dict(ckpt["ema_state_dict"])
        print(
            f"[DETR] Resumed from epoch {start_epoch}, "
            f"best mAP={best_map:.4f}"
        )

    effective_bs = batch_size * grad_accum
    print(
        f"[DETR] Training {epochs} epochs, batch={batch_size}, "
        f"grad_accum={grad_accum}, effective_bs={effective_bs}, "
        f"warmup={warmup_epochs}"
    )

    for epoch in range(start_epoch, epochs + 1):
        train_loss = train_one_epoch(
            model, optimizer, train_loader, device, epoch,
            scaler=scaler, clip_norm=cfg["training"]["clip_norm"],
            metric_tracker=metric_tracker,
            gradient_accumulation_steps=grad_accum,
        )
        lr_scheduler.step()
        current_lr_bb = optimizer.param_groups[0]["lr"]
        current_lr_tr = optimizer.param_groups[1]["lr"]

        ema.update(model)

        print(f"\n[DETR] Validation after epoch {epoch}...")
        coco_eval = evaluate(model, val_loader, device)

        current_map = 0.0
        if coco_eval is not None:
            current_map = coco_eval.stats[0]
            val_metrics = {
                "epoch": epoch,
                "train_loss": train_loss,
                "mAP@0.5:0.95": coco_eval.stats[0],
                "mAP@0.5": coco_eval.stats[1],
                "mAP@0.75": coco_eval.stats[2],
                "AR@10": coco_eval.stats[7],
                "lr_backbone": current_lr_bb,
                "lr_transformer": current_lr_tr,
            }
            wandb.log(val_metrics)
            metric_tracker.update(**val_metrics)

            if current_map > best_map:
                best_map = current_map
                best_epoch = epoch
                patience_counter = 0
                save_checkpoint(
                    model, optimizer, epoch,
                    {"best_map": best_map, "best_epoch": best_epoch},
                    f"{results_dir}/weights/best_model.pth",
                    scaler=scaler, scheduler=lr_scheduler,
                    extra={"ema_state_dict": ema.state_dict()},
                )
                torch.save(
                    ema.state_dict(),
                    f"{results_dir}/weights/ema_model.pth",
                )
                print(
                    f"  New best model! mAP={best_map:.4f} (epoch {epoch})"
                )
            else:
                patience_counter += 1

        if epoch % cfg["training"]["save_every"] == 0:
            save_checkpoint(
                model, optimizer, epoch,
                {"train_loss": train_loss, "mAP": current_map},
                f"{results_dir}/weights/epoch_{epoch:03d}.pth",
                scaler=scaler, scheduler=lr_scheduler,
            )

        save_checkpoint(
            model, optimizer, epoch,
            {"train_loss": train_loss, "mAP": current_map},
            f"{results_dir}/weights/last_checkpoint.pth",
            scaler=scaler, scheduler=lr_scheduler,
            extra={
                "best_map": best_map,
                "best_epoch": best_epoch,
                "ema_state_dict": ema.state_dict(),
            },
        )

        if patience_counter >= cfg["training"]["early_stop_patience"]:
            print(
                f"[DETR] Early stopping at epoch {epoch} "
                f"(no improvement for {patience_counter} epochs)"
            )
            break

        print(
            f"  LR backbone: {current_lr_bb:.2e} | "
            f"LR transformer: {current_lr_tr:.2e} | "
            f"Patience: {patience_counter}/{cfg['training']['early_stop_patience']}"
        )
        print()

    plot_training_curves(metric_tracker, f"{results_dir}/curves")
    with open(f"{results_dir}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    print(
        f"\n[DETR] Training complete. "
        f"Best mAP: {best_map:.4f} at epoch {best_epoch}"
    )
    return best_map, best_epoch


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_model(cfg):
    results_dir = cfg["results_dir"]
    print("\n[DETR] Evaluating best model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg["training"]["batch_size"],
        shuffle=False, collate_fn=collate_fn,
        num_workers=cfg["data"]["num_workers"],
    )

    model = build_detr(cfg)
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        model.load_state_dict(
            torch.load(ema_path, map_location=device, weights_only=True)
        )
        print("  Loaded EMA weights")
    elif os.path.exists(best_path):
        model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
        print("  Loaded best model weights")
    model.to(device)
    model.eval()

    coco_eval = evaluate(
        model, val_loader, device,
        output_file=f"{results_dir}/val_preds.json",
    )

    if coco_eval is not None:
        metrics = {
            "mAP@0.5:0.95": coco_eval.stats[0],
            "mAP@0.5": coco_eval.stats[1],
            "mAP@0.75": coco_eval.stats[2],
            "mAP_small": coco_eval.stats[3],
            "mAP_medium": coco_eval.stats[4],
            "mAP_large": coco_eval.stats[5],
            "AR@1": coco_eval.stats[6],
            "AR@10": coco_eval.stats[7],
            "AR@100": coco_eval.stats[8],
        }
        with open(f"{results_dir}/metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"[DETR] Metrics saved to {results_dir}/metrics.json")

    print("\n[DETR] Running TTA evaluation...")
    coco_eval_tta = evaluate(
        model, val_loader, device,
        output_file=f"{results_dir}/val_preds_tta.json", tta=True,
    )
    if coco_eval_tta is not None:
        metrics_tta = {
            "mAP@0.5:0.95": coco_eval_tta.stats[0],
            "mAP@0.5": coco_eval_tta.stats[1],
            "mAP@0.75": coco_eval_tta.stats[2],
        }
        with open(f"{results_dir}/metrics_tta.json", "w") as f:
            json.dump(metrics_tta, f, indent=2)
        print(f"[DETR] TTA metrics saved to {results_dir}/metrics_tta.json")

    confusion, class_results = compute_confusion_matrix(
        model, val_loader, device,
    )
    save_confusion_matrix_plot(
        confusion, f"{results_dir}/confusion_matrix.png",
    )

    test_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    if os.path.isfile(test_ann):
        test_dataset = CocoDetection(
            cfg["data"].get("test_images", "dataset/coco/test"),
            test_ann,
            transforms=AugmentedTransform(train=False, cfg=cfg),
        )
        test_loader = DataLoader(
            test_dataset, batch_size=cfg["training"]["batch_size"],
            shuffle=False, collate_fn=collate_fn,
            num_workers=cfg["data"]["num_workers"],
        )
        evaluate_test(
            model, test_loader, device,
            f"{results_dir}/test_preds.json",
            model_name=cfg.get("model", {}).get("name", "DETR-ResNet50"),
        )
    else:
        print("[TEST] No annotations found. Running inference-only prediction.")


# =============================================================================
# XAI — DETR Attention with Query Visualization
# =============================================================================
def run_xai(cfg):
    results_dir = cfg["results_dir"]
    print("\n[DETR] Generating XAI attention maps...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_detr(cfg)
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        model.load_state_dict(
            torch.load(ema_path, map_location=device, weights_only=True)
        )
    elif os.path.exists(best_path):
        model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
    model.to(device)
    model.eval()

    xai_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    xai_img_dir = cfg["data"].get("test_images", "dataset/coco/test")
    if not os.path.isfile(xai_ann):
        xai_ann = cfg["data"].get("val_ann", "dataset/coco/val.json")
        xai_img_dir = cfg["data"]["val_images"]
    if not os.path.exists(xai_ann):
        print("[DETR] No dataset available for XAI.")
        return

    xai_dataset = CocoDetection(
        xai_img_dir, xai_ann,
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    loader = DataLoader(
        xai_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_fn, num_workers=2,
    )

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image as PILImage, ImageDraw

    count = 0
    max_samples = 5

    for images, targets in loader:
        if count >= max_samples:
            break
        image = images[0].unsqueeze(0).to(device)

        extractor = DETRAttentionExtractor(model)
        heatmaps, meta = extractor.extract(image)
        extractor.remove_hook()

        if not heatmaps or len(heatmaps) == 0:
            continue

        img_np = image.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        model.eval()
        with torch.no_grad():
            outputs = model(image)
        boxes = outputs[0]["boxes"].cpu().numpy()
        scores = outputs[0]["scores"].cpu().numpy()
        labels = outputs[0]["labels"].cpu().numpy()

        gt_boxes = targets[0]["boxes"].numpy()
        gt_labels = targets[0]["labels"].numpy()

        n_queries_show = min(len(heatmaps), 6)
        n_cols = 2 + n_queries_show
        fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
        if n_cols == 1:
            axes = [axes]

        axes[0].imshow(img_np)
        draw_img = PILImage.fromarray(img_np.copy())
        draw = ImageDraw.Draw(draw_img)
        for box, sc, lb in zip(boxes, scores, labels):
            if sc < 0.3:
                continue
            x1, y1, x2, y2 = box[:4]
            color = CLASS_COLORS.get(int(lb), (255, 255, 0))
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text(
                (x1 + 2, y1 - 16),
                f'{CLASS_NAMES.get(int(lb), str(lb))}: {sc:.2f}',
                fill=color,
            )
        axes[0].imshow(np.array(draw_img))
        axes[0].set_title("DETR — Detections", fontsize=10)
        axes[0].axis("off")

        gt_img = PILImage.fromarray(img_np.copy())
        gt_draw = ImageDraw.Draw(gt_img)
        for box, lb in zip(gt_boxes, gt_labels):
            x1, y1, x2, y2 = box[:4]
            color = CLASS_COLORS.get(int(lb), (255, 255, 0))
            gt_draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            gt_draw.text(
                (x1 + 2, y1 - 14),
                f'GT: {CLASS_NAMES.get(int(lb), str(lb))}',
                fill=color,
            )
        axes[1].imshow(np.array(gt_img))
        axes[1].set_title("Ground Truth", fontsize=10)
        axes[1].axis("off")

        for qi in range(n_queries_show):
            attn = heatmaps[qi]
            attn_resized = np.array(
                PILImage.fromarray((attn * 255).astype(np.uint8)).resize(
                    (img_np.shape[1], img_np.shape[0]), PILImage.BILINEAR
                )
            ) / 255.0
            cmap = plt.get_cmap("jet")
            attn_colored = cmap(attn_resized)[:, :, :3]
            combined = (1 - 0.5) * img_np / 255 + 0.5 * attn_colored
            axes[2 + qi].imshow(np.clip(combined, 0, 1))

            m = meta[qi] if qi < len(meta) else {}
            label_name = CLASS_NAMES.get(m.get("label", -1), "bg")
            score_val = m.get("score", 0.0)
            axes[2 + qi].set_title(
                f'Q{qi}: {label_name}\n{score_val:.2f}', fontsize=9,
            )
            axes[2 + qi].axis("off")

        plt.suptitle(
            f"DETR Attention Analysis (epoch={cfg['training']['epochs']})",
            fontsize=12,
        )
        plt.tight_layout()
        out_path = f"{results_dir}/explain/sample_{count}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved {out_path}")

        query_info = []
        for qi in range(min(len(heatmaps), len(meta))):
            m = meta[qi]
            query_info.append({
                "query": qi,
                "label": CLASS_NAMES.get(m["label"], "unknown"),
                "score": m["score"],
                "box": m["box"],
            })
        with open(f"{results_dir}/explain/queries_{count}.json", "w") as f:
            json.dump(query_info, f, indent=2)

        count += 1

    print(f"[DETR] XAI complete. {count} samples analyzed.")


# =============================================================================
# Main
# =============================================================================
def main():
    cfg = get_config()
    train(cfg)
    evaluate_model(cfg)
    run_xai(cfg)


if __name__ == "__main__":
    main()


### 14. Write `train_efficientdet.py`

In [ ]:
%%writefile train_efficientdet.py
#!/usr/bin/env python3
"""EfficientDet-D2 Training Pipeline — Research-Grade Implementation for TBX11K.

Usage:
    python train_efficientdet.py
    python train_efficientdet.py --epochs 50 --lr 1e-4
    python train_efficientdet.py --resume results/efficientdet/weights/last_checkpoint.pth
    python train_efficientdet.py --batch-size 4
"""

import os
import sys
import json
import copy
import argparse

import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF

try:
    from effdet import create_model
    from effdet.efficientdet import DetBenchTrain
    HAS_EFFDET = True
except ImportError:
    HAS_EFFDET = False

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass
from utils.coco_dataset import CocoDetection, collate_fn
from utils.engine import (
    set_seed, MetricTracker, save_checkpoint, load_checkpoint,
    evaluate, evaluate_test,
    compute_confusion_matrix, save_confusion_matrix_plot,
    plot_training_curves,
)
from utils.ema import ModelEMA
from explain.gradcam import GradCAM, get_target_layer
from explain.visualize import overlay_heatmap, draw_detections

CLASS_NAMES = {
    0: "Background", 1: "ActiveTuberculosis", 2: "ObsoletePulmonaryTuberculosis",
}
NUM_CLASSES = 3
IMG_SIZE = 768
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# =============================================================================
# Configuration
# =============================================================================
DEFAULT_CONFIG = {
    "model": {
        "name": "tf_efficientdet_d2",
        "num_classes": NUM_CLASSES,
        "img_size": IMG_SIZE,
    },
    "training": {
        "epochs": 100,
        "batch_size": 4,
        "lr": 1e-4,
        "lr_backbone": 1e-5,
        "weight_decay": 1e-4,
        "clip_norm": 1.0,
        "warmup_epochs": 5,
        "ema_decay": 0.99,
        "early_stop_patience": 15,
        "save_every": 10,
        "seed": 42,
        "resume": None,
    },
    "augmentation": {
        "hflip_prob": 0.5,
        "brightness": 0.3,
        "contrast": 0.3,
        "gamma": 0.2,
        "noise_std": 0.05,
        "normalize": True,
    },
    "data": {
        "train_images": "dataset/coco/train",
        "val_images": "dataset/coco/val",
        "train_ann": "dataset/coco/train.json",
        "val_ann": "dataset/coco/val.json",
        "test_images": "dataset/coco/test",
        "test_ann": "dataset/coco/test.json",
        "num_workers": 2,
    },
    "results_dir": "results/efficientdet",
}


# =============================================================================
# CLI Argument Parser
# =============================================================================
def parse_args():
    p = argparse.ArgumentParser(description="EfficientDet-D2 TB Detection Training")
    p.add_argument("--epochs", type=int, default=None)
    p.add_argument("--batch-size", type=int, default=None)
    p.add_argument("--lr", type=float, default=None)
    p.add_argument("--lr-backbone", type=float, default=None)
    p.add_argument("--weight-decay", type=float, default=None)
    p.add_argument("--clip-norm", type=float, default=None)
    p.add_argument("--warmup-epochs", type=int, default=None)
    p.add_argument("--ema-decay", type=float, default=None)
    p.add_argument("--early-stop-patience", type=int, default=None)
    p.add_argument("--save-every", type=int, default=None)
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--resume", type=str, default=None)
    p.add_argument("--results-dir", type=str, default=None)
    args, _ = p.parse_known_args()
    return args


def get_config():
    args = parse_args()
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    t = cfg["training"]
    if args.epochs is not None: t["epochs"] = args.epochs
    if args.batch_size is not None: t["batch_size"] = args.batch_size
    if args.lr is not None: t["lr"] = args.lr
    if args.lr_backbone is not None: t["lr_backbone"] = args.lr_backbone
    if args.weight_decay is not None: t["weight_decay"] = args.weight_decay
    if args.clip_norm is not None: t["clip_norm"] = args.clip_norm
    if args.warmup_epochs is not None: t["warmup_epochs"] = args.warmup_epochs
    if args.ema_decay is not None: t["ema_decay"] = args.ema_decay
    if args.early_stop_patience is not None:
        t["early_stop_patience"] = args.early_stop_patience
    if args.save_every is not None: t["save_every"] = args.save_every
    if args.seed is not None: t["seed"] = args.seed
    if args.resume is not None: t["resume"] = args.resume
    if args.results_dir is not None: cfg["results_dir"] = args.results_dir
    return cfg


# =============================================================================
# Data Augmentation Transforms
# =============================================================================
class AugmentedTransform:
    def __init__(self, train=True, cfg=None):
        self.train = train
        aug = (cfg or {}).get("augmentation", {})
        self.hflip_prob = aug.get("hflip_prob", 0.5)
        self.brightness = aug.get("brightness", 0.3)
        self.contrast = aug.get("contrast", 0.3)
        self.gamma_range = aug.get("gamma", 0.2)
        self.noise_std = aug.get("noise_std", 0.05)
        self.normalize = aug.get("normalize", True)

    def __call__(self, image, target):
        image = TF.to_tensor(image)

        if self.train:
            if torch.rand(1).item() < self.hflip_prob:
                image = TF.hflip(image)
                if len(target["boxes"]) > 0:
                    boxes = target["boxes"].clone()
                    w = image.shape[-1]
                    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                    target["boxes"] = boxes

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.brightness
                image = TF.adjust_brightness(image, factor)

            if torch.rand(1).item() < 0.5:
                factor = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.contrast
                image = TF.adjust_contrast(image, factor)

            if torch.rand(1).item() < 0.3:
                g = 1.0 + (torch.rand(1).item() - 0.5) * 2 * self.gamma_range
                image = image.clamp(min=1e-6).pow(g)

            if torch.rand(1).item() < 0.3:
                image = (image + torch.randn_like(image) * self.noise_std).clamp(0, 1)

        if self.normalize:
            image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)

        return image, target


# =============================================================================
# EfficientDet Collate — resizes images AND scales bounding boxes
# =============================================================================
def make_collate_fn(img_size=IMG_SIZE):
    def collate_effdet(batch):
        images, targets = zip(*batch)
        orig_h, orig_w = images[0].shape[1], images[0].shape[2]
        scale_x = img_size / orig_w
        scale_y = img_size / orig_h

        resized = []
        scaled_targets = []
        for img, target in zip(images, targets):
            resized.append(
                TF.interpolate(
                    img.unsqueeze(0), size=(img_size, img_size),
                    mode="bilinear", align_corners=False,
                ).squeeze(0)
            )
            new_target = {}
            for k, v in target.items():
                if k == "boxes" and len(v) > 0:
                    boxes = v.clone()
                    boxes[:, [0, 2]] *= scale_x
                    boxes[:, [1, 3]] *= scale_y
                    new_target[k] = boxes
                else:
                    new_target[k] = v
            scaled_targets.append(new_target)

        images = torch.stack(resized)

        cls_list = [t["labels"] for t in scaled_targets]
        box_list = [t["boxes"] for t in scaled_targets]
        device = box_list[0].device if len(box_list) > 0 else torch.device("cpu")
        batch_size = len(cls_list)
        max_boxes = max((len(c) for c in cls_list), default=1)
        max_boxes = max(max_boxes, 1)

        cls_pad = torch.zeros(batch_size, max_boxes, dtype=torch.int64, device=device)
        box_pad = torch.zeros(batch_size, max_boxes, 4, dtype=torch.float32, device=device)

        for i in range(batch_size):
            n = min(len(cls_list[i]), max_boxes)
            if n > 0:
                cls_pad[i, :n] = cls_list[i][:n]
                box_pad[i, :n] = box_list[i][:n]

        return images, {
            "cls": cls_pad,
            "bbox": box_pad,
            "img_scale": torch.ones(batch_size, 1, device=device),
            "img_size": torch.full(
                (batch_size, 2), img_size, dtype=torch.float32, device=device,
            ),
        }

    return collate_effdet


# =============================================================================
# EfficientDet → TorchVision output adapter
# =============================================================================
class EfficientDetWrapper(nn.Module):
    """Wraps raw EfficientDet to return TorchVision-format list of dicts."""

    def __init__(self, model, score_thresh=0.05):
        super().__init__()
        self.model = model
        self.score_thresh = score_thresh

    def forward(self, images):
        with torch.no_grad():
            detections = self.model(images)
        results = []
        for i in range(detections.shape[0]):
            dets = detections[i]
            valid = dets[:, 4] > self.score_thresh
            dets = dets[valid]
            results.append({
                "boxes": dets[:, :4],
                "scores": dets[:, 4],
                "labels": dets[:, 5].int(),
            })
        return results


# =============================================================================
# Build EfficientDet
# =============================================================================
def build_efficientdet(cfg):
    if not HAS_EFFDET:
        raise ImportError(
            "effdet not installed. Install with: pip install effdet"
        )
    num_classes = cfg["model"]["num_classes"]
    use_pretrained = False
    try:
        raw_model = create_model(
            "tf_efficientdet_d2", pretrained=True, num_classes=num_classes,
        )
        use_pretrained = True
        print("  Loaded COCO-pretrained EfficientDet-D2")
    except Exception:
        import warnings
        warnings.warn("Could not load pretrained weights, using random init. Disabling AMP.")
        raw_model = create_model(
            "tf_efficientdet_d2", pretrained=False, num_classes=num_classes,
        )
    return raw_model, use_pretrained


# =============================================================================
# Training
# =============================================================================
def train(cfg):
    results_dir = cfg["results_dir"]
    os.makedirs(f"{results_dir}/weights", exist_ok=True)
    os.makedirs(f"{results_dir}/curves", exist_ok=True)
    os.makedirs(f"{results_dir}/explain", exist_ok=True)

    set_seed(cfg["training"]["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[EfficientDet] Device: {device}")

    wandb.init(
        mode=os.environ.get("WANDB_MODE", "online"), project="tbx11k", name="efficientdet",
        config=cfg,
    )

    img_size = cfg["model"]["img_size"]
    collate_train = make_collate_fn(img_size)
    collate_val = make_collate_fn(img_size)

    train_dataset = CocoDetection(
        cfg["data"]["train_images"], cfg["data"]["train_ann"],
        transforms=AugmentedTransform(train=True, cfg=cfg),
    )
    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )

    batch_size = cfg["training"]["batch_size"]

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_train, num_workers=cfg["data"]["num_workers"],
        drop_last=True, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_val, num_workers=cfg["data"]["num_workers"],
        pin_memory=True,
    )

    raw_model, use_pretrained = build_efficientdet(cfg)
    raw_model.to(device)
    bench_train = DetBenchTrain(raw_model)

    ema = ModelEMA(raw_model, decay=cfg["training"]["ema_decay"])

    params = [p for p in raw_model.parameters() if p.requires_grad]
    backbone_params = [p for n, p in raw_model.named_parameters()
                       if "backbone" in n and p.requires_grad]
    head_params = [p for n, p in raw_model.named_parameters()
                   if "backbone" not in n and p.requires_grad]

    lr_main = cfg["training"]["lr"]
    lr_bb = cfg["training"]["lr_backbone"]
    wd = cfg["training"]["weight_decay"]

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": lr_bb, "name": "backbone"},
        {"params": head_params, "lr": lr_main, "name": "head"},
    ], weight_decay=wd)

    print(f"  Optimizer: backbone lr={lr_bb:.1e}, head lr={lr_main:.1e}, wd={wd}")

    epochs = cfg["training"]["epochs"]
    warmup_epochs = cfg["training"]["warmup_epochs"]
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, total_iters=warmup_epochs,
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=1e-8,
    )
    lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, [warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs],
    )

    scaler = None
    metric_tracker = MetricTracker()

    start_epoch = 1
    best_map = 0.0
    best_epoch = -1
    patience_counter = 0

    if cfg["training"]["resume"] and os.path.exists(cfg["training"]["resume"]):
        ckpt = load_checkpoint(
            cfg["training"]["resume"], raw_model, optimizer, scaler, lr_scheduler,
        )
        start_epoch = ckpt.get("epoch", 0) + 1
        best_map = ckpt.get("metrics", {}).get("best_map", 0.0)
        best_epoch = ckpt.get("metrics", {}).get("best_epoch", -1)
        if "ema_state_dict" in ckpt:
            ema.load_state_dict(ckpt["ema_state_dict"])
        print(
            f"[EfficientDet] Resumed from epoch {start_epoch}, "
            f"best mAP={best_map:.4f}"
        )

    print(
        f"[EfficientDet] Training {epochs} epochs, batch={batch_size}, "
        f"img_size={img_size}, warmup={warmup_epochs}"
    )

    for epoch in range(start_epoch, epochs + 1):
        raw_model.train()
        total_loss = 0.0
        num_batches = 0
        start_time = __import__("time").time()

        for i, (images, targets) in enumerate(train_loader):
            images = images.to(device)
            for k in targets:
                if isinstance(targets[k], torch.Tensor):
                    targets[k] = targets[k].to(device)

            optimizer.zero_grad()

            if scaler is not None:
                with torch.amp.autocast(device_type="cuda"):
                    out = bench_train(images, targets)
                    loss = out["loss"]
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    raw_model.parameters(), cfg["training"]["clip_norm"],
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                out = bench_train(images, targets)
                loss = out["loss"]
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    raw_model.parameters(), cfg["training"]["clip_norm"],
                )
                optimizer.step()

            total_loss += loss.item()
            num_batches += 1

            if i % 50 == 0 and num_batches > 0:
                lr = optimizer.param_groups[1]["lr"]
                elapsed = __import__("time").time() - start_time
                parts = [
                    f"Epoch [{epoch}] Step [{i}/{len(train_loader)}]",
                    f"Loss: {loss.item():.4f}",
                    f"LR: {lr:.2e}",
                    f"Time: {elapsed:.1f}s",
                ]
                print(" | ".join(parts))
                wandb.log({
                    "step_loss": loss.item(),
                    "step": i + epoch * len(train_loader),
                    "learning_rate": lr,
                })

        lr_scheduler.step()
        avg_loss = total_loss / max(num_batches, 1)
        current_lr_bb = optimizer.param_groups[0]["lr"]
        current_lr_main = optimizer.param_groups[1]["lr"]

        ema.update(raw_model)

        print(f"\n[Evaluate] Validation after epoch {epoch}...")
        wrapper = EfficientDetWrapper(raw_model)
        wrapper_ema = EfficientDetWrapper(ema.model)
        coco_eval = evaluate(wrapper_ema, val_loader, device)

        current_map = 0.0
        if coco_eval is not None:
            current_map = coco_eval.stats[0]
            val_metrics = {
                "epoch": epoch,
                "train_loss": avg_loss,
                "mAP@0.5:0.95": coco_eval.stats[0],
                "mAP@0.5": coco_eval.stats[1],
                "mAP@0.75": coco_eval.stats[2],
                "AR@10": coco_eval.stats[7],
                "lr_backbone": current_lr_bb,
                "lr_head": current_lr_main,
            }
            wandb.log(val_metrics)
            metric_tracker.update(**val_metrics)

            if current_map > best_map:
                best_map = current_map
                best_epoch = epoch
                patience_counter = 0
                save_checkpoint(
                    raw_model, optimizer, epoch,
                    {"best_map": best_map, "best_epoch": best_epoch},
                    f"{results_dir}/weights/best_model.pth",
                    scaler=scaler, scheduler=lr_scheduler,
                    extra={"ema_state_dict": ema.state_dict()},
                )
                torch.save(
                    ema.state_dict(),
                    f"{results_dir}/weights/ema_model.pth",
                )
                print(
                    f"  New best model! mAP={best_map:.4f} (epoch {epoch})"
                )
            else:
                patience_counter += 1

        if epoch % cfg["training"]["save_every"] == 0:
            save_checkpoint(
                raw_model, optimizer, epoch,
                {"train_loss": avg_loss, "mAP": current_map},
                f"{results_dir}/weights/epoch_{epoch:03d}.pth",
                scaler=scaler, scheduler=lr_scheduler,
            )

        save_checkpoint(
            raw_model, optimizer, epoch,
            {"train_loss": avg_loss, "mAP": current_map},
            f"{results_dir}/weights/last_checkpoint.pth",
            scaler=scaler, scheduler=lr_scheduler,
            extra={
                "best_map": best_map,
                "best_epoch": best_epoch,
                "ema_state_dict": ema.state_dict(),
            },
        )

        if patience_counter >= cfg["training"]["early_stop_patience"]:
            print(
                f"[EfficientDet] Early stopping at epoch {epoch} "
                f"(no improvement for {patience_counter} epochs)"
            )
            break

        print(
            f"  LR backbone: {current_lr_bb:.2e} | "
            f"LR head: {current_lr_main:.2e} | "
            f"Patience: {patience_counter}/{cfg['training']['early_stop_patience']}"
        )
        print()

    plot_training_curves(metric_tracker, f"{results_dir}/curves")
    with open(f"{results_dir}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    print(
        f"\n[EfficientDet] Training complete. "
        f"Best mAP: {best_map:.4f} at epoch {best_epoch}"
    )
    return best_map, best_epoch


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_model(cfg):
    results_dir = cfg["results_dir"]
    print("\n[EfficientDet] Evaluating best model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    img_size = cfg["model"]["img_size"]
    collate_val = make_collate_fn(img_size)

    val_dataset = CocoDetection(
        cfg["data"]["val_images"], cfg["data"]["val_ann"],
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg["training"]["batch_size"],
        shuffle=False, collate_fn=collate_val,
        num_workers=cfg["data"]["num_workers"],
    )

    raw_model = build_efficientdet(cfg)
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        state = torch.load(ema_path, map_location=device, weights_only=True)
        if isinstance(state, dict) and "model_state_dict" in state:
            raw_model.load_state_dict(state["model_state_dict"])
        else:
            raw_model.load_state_dict(state)
        print("  Loaded EMA weights")
    elif os.path.exists(best_path):
        raw_model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
        print("  Loaded best model weights")
    raw_model.to(device)
    raw_model.eval()

    wrapper = EfficientDetWrapper(raw_model)

    coco_eval = evaluate(
        wrapper, val_loader, device,
        output_file=f"{results_dir}/val_preds.json",
    )

    if coco_eval is not None:
        metrics = {
            "mAP@0.5:0.95": coco_eval.stats[0],
            "mAP@0.5": coco_eval.stats[1],
            "mAP@0.75": coco_eval.stats[2],
            "mAP_small": coco_eval.stats[3],
            "mAP_medium": coco_eval.stats[4],
            "mAP_large": coco_eval.stats[5],
            "AR@1": coco_eval.stats[6],
            "AR@10": coco_eval.stats[7],
            "AR@100": coco_eval.stats[8],
        }
        with open(f"{results_dir}/metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"[EfficientDet] Metrics saved to {results_dir}/metrics.json")

    print("\n[EfficientDet] Running TTA evaluation...")
    coco_eval_tta = evaluate(
        wrapper, val_loader, device,
        output_file=f"{results_dir}/val_preds_tta.json", tta=True,
    )
    if coco_eval_tta is not None:
        metrics_tta = {
            "mAP@0.5:0.95": coco_eval_tta.stats[0],
            "mAP@0.5": coco_eval_tta.stats[1],
            "mAP@0.75": coco_eval_tta.stats[2],
        }
        with open(f"{results_dir}/metrics_tta.json", "w") as f:
            json.dump(metrics_tta, f, indent=2)
        print(
            f"[EfficientDet] TTA metrics saved to {results_dir}/metrics_tta.json"
        )

    confusion, class_results = compute_confusion_matrix(
        wrapper, val_loader, device,
    )
    save_confusion_matrix_plot(
        confusion, f"{results_dir}/confusion_matrix.png",
    )

    test_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    if os.path.isfile(test_ann):
        test_dataset = CocoDetection(
            cfg["data"].get("test_images", "dataset/coco/test"),
            test_ann,
            transforms=AugmentedTransform(train=False, cfg=cfg),
        )
        test_loader = DataLoader(
            test_dataset, batch_size=cfg["training"]["batch_size"],
            shuffle=False, collate_fn=collate_val,
            num_workers=cfg["data"]["num_workers"],
        )
        evaluate_test(
            wrapper, test_loader, device,
            f"{results_dir}/test_preds.json",
            model_name=cfg.get("model", {}).get("name", "EfficientDet-D2"),
        )
    else:
        print("[TEST] No annotations found. Running inference-only prediction.")


# =============================================================================
# XAI — Grad-CAM
# =============================================================================
def run_xai(cfg):
    results_dir = cfg["results_dir"]
    print("\n[EfficientDet] Generating XAI explanations...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    raw_model = build_efficientdet(cfg)
    ema_path = f"{results_dir}/weights/ema_model.pth"
    best_path = f"{results_dir}/weights/best_model.pth"
    if os.path.exists(ema_path):
        state = torch.load(ema_path, map_location=device, weights_only=True)
        if isinstance(state, dict) and "model_state_dict" in state:
            raw_model.load_state_dict(state["model_state_dict"])
        else:
            raw_model.load_state_dict(state)
    elif os.path.exists(best_path):
        raw_model.load_state_dict(
            torch.load(best_path, map_location=device, weights_only=True)
        )
    raw_model.to(device)
    raw_model.eval()

    xai_ann = cfg["data"].get("test_ann", "dataset/coco/test.json")
    xai_img_dir = cfg["data"].get("test_images", "dataset/coco/test")
    if not os.path.isfile(xai_ann):
        xai_ann = cfg["data"].get("val_ann", "dataset/coco/val.json")
        xai_img_dir = cfg["data"]["val_images"]
    if not os.path.exists(xai_ann):
        print("[EfficientDet] No dataset available for XAI.")
        return

    img_size = cfg["model"]["img_size"]
    collate_val = make_collate_fn(img_size)
    xai_dataset = CocoDetection(
        xai_img_dir, xai_ann,
        transforms=AugmentedTransform(train=False, cfg=cfg),
    )
    loader = DataLoader(
        xai_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_val, num_workers=2,
    )

    wrapper = EfficientDetWrapper(raw_model, score_thresh=0.3)

    target_layer = raw_model.backbone.conv_head
    gradcam = GradCAM(raw_model, target_layer)

    count = 0
    max_samples = 5

    for images, targets in loader:
        if count >= max_samples:
            break
        image = images[0].unsqueeze(0).to(device)

        with torch.set_grad_enabled(True):
            heatmap = gradcam.generate(image)

        if heatmap is not None:
            overlay = overlay_heatmap(image, heatmap)
            outputs = wrapper(image)
            if outputs and len(outputs[0]["boxes"]) > 0:
                detections = {
                    "boxes": outputs[0]["boxes"].cpu().numpy(),
                    "scores": outputs[0]["scores"].cpu().numpy(),
                    "labels": outputs[0]["labels"].cpu().numpy(),
                }
                overlay = draw_detections(overlay, detections)
                overlay.save(f"{results_dir}/explain/sample_{count}.png")
                print(f"  Saved {results_dir}/explain/sample_{count}.png")
                count += 1

    gradcam.remove_hooks()
    print("[EfficientDet] XAI complete.")


# =============================================================================
# Main
# =============================================================================
def main():
    if not HAS_EFFDET:
        print("[SKIP] effdet not installed — install with: pip install effdet")
        return
    cfg = get_config()
    train(cfg)
    evaluate_model(cfg)
    run_xai(cfg)


if __name__ == "__main__":
    main()


### 15. Cache backbone weights

In [ ]:
# Pre-download backbone weights to avoid timeout during training
import torch
url = "https://download.pytorch.org/models/resnet50-11ad3fa6.pth"
torch.hub.load_state_dict_from_url(url, map_location="cpu")
print("Backbone weights cached successfully")

## 16. Convert Annotations + Run EDA

In [ ]:
# Cell 15: Convert Annotations + Run EDA
%cd /content/OBJECT_DETECTION
!python convert.py --data-root Images --output-dir dataset
!python eda.py

from IPython.display import Image as DImg, display
for fname in ['tag_distribution.png', 'bbox_class_distribution.png', 'bbox_spatial_heatmap.png',
              'bbox_size_distribution.png', 'bbox_aspect_ratio.png', 'boxes_per_image.png', 'sample_grid.png']:
    path = f'results/eda/{fname}'
    if os.path.exists(path):
        display(DImg(path))

## 17. Train FCOS

In [ ]:
# Cell 16: Train FCOS
%cd /content/OBJECT_DETECTION
!mkdir -p results/fcos && python train_fcos.py 2>&1 | tee results/fcos/training_log.txt
print('FCOS complete.')
!cp -r results /content/drive/MyDrive/OBJECT_DETECTION/results_fcos
print('Results saved to Drive.')


## 18. Train RetinaNet

In [ ]:
# Cell 18: Train RetinaNet
%cd /content/OBJECT_DETECTION
!mkdir -p results/retinanet && python train_retinanet.py 2>&1 | tee results/retinanet/training_log.txt
print('RetinaNet complete.')
!cp -r results /content/drive/MyDrive/OBJECT_DETECTION/results_retinanet
print('Results saved to Drive.')


### 19. Install effdet (before EfficientDet)

In [ ]:
!pip install -q effdet

## 20. Train EfficientDet-D2

In [ ]:
# Cell 17: Train EfficientDet-D2
%cd /content/OBJECT_DETECTION
!mkdir -p results/efficientdet && python train_efficientdet.py 2>&1 | tee results/efficientdet/training_log.txt
print('EfficientDet complete.')
!cp -r results /content/drive/MyDrive/OBJECT_DETECTION/results_efficientdet
print('Results saved to Drive.')


## 21. Train DETR

In [ ]:
# Cell 19: Train DETR
%cd /content/OBJECT_DETECTION
!mkdir -p results/detr && python train_detr.py 2>&1 | tee results/detr/training_log.txt
print('DETR complete.')
!cp -r results /content/drive/MyDrive/OBJECT_DETECTION/results_detr
print('Results saved to Drive.')


## 22. Model Comparison

In [ ]:
# Cell 20: Model Comparison
%cd /content/OBJECT_DETECTION
import os, json, pandas as pd, matplotlib.pyplot as plt

models = ['fcos', 'efficientdet', 'retinanet', 'detr']
labels = ['FCOS', 'EfficientDet-D2', 'RetinaNet', 'DETR']

all_metrics = {}
for m in models:
    p = f'results/{m}/metrics.json'
    if os.path.exists(p):
        with open(p) as f:
            all_metrics[m] = json.load(f)

df = pd.DataFrame(all_metrics).T
df.index = labels
print('\n=== MODEL COMPARISON ===')
print(df.round(4).to_string())
df.round(4).to_csv('results/comparison.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if 'mAP@0.5:0.95' in df.columns:
    df['mAP@0.5:0.95'].plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    axes[0].set_ylabel('mAP@0.5:0.95')
    axes[0].set_title('Detection Performance')
if 'AP_ActiveTB' in df.columns and 'AP_ObsoleteTB' in df.columns:
    df[['AP_ActiveTB', 'AP_ObsoleteTB']].plot(kind='bar', ax=axes[1])
    axes[1].set_ylabel('AP')
    axes[1].set_title('Per-Class AP')
plt.tight_layout()
plt.savefig('results/comparison_barplot.png', dpi=150)
plt.show()

## 23. Confusion Matrices + XAI

In [ ]:
# Cell 21: Confusion Matrices + XAI
%cd /content/OBJECT_DETECTION
from IPython.display import Image as DImg, display

for name, label in zip(['fcos','efficientdet','retinanet','detr'],
                       ['FCOS','EfficientDet-D2','RetinaNet','DETR']):
    cm = f'results/{name}/confusion_matrix.png'
    if os.path.exists(cm):
        print(f'\n{label}:')
        display(DImg(cm))
    xdir = f'results/{name}/explain'
    if os.path.isdir(xdir):
        for s in sorted(os.listdir(xdir))[:2]:
            display(DImg(os.path.join(xdir, s)))

### 24. Save Results to Google Drive

In [ ]:
# Cell: Save Results to Drive
import os
drive_path = '/content/drive/MyDrive/OBJECT_DETECTION/results'
!cp -r results "$drive_path"
print(f'Results saved to {drive_path}')
